# YOLOv8m Cross-Year AF-MCSO: 600-Unit HPO (GitHub Version)

This GitHub-ready notebook implements the updated **Adaptive Fidelity-Aware Modified Cuckoo Search (AF-MCSO)** method for both cross-year weed detection directions: **2021 → 2022** and **2022 → 2021**.

- **Search:** 4 initial 20-epoch configurations, 2 full-fidelity confirmations, 5 generations of 2×20-epoch screening plus 1×40-epoch promotion, and a final 40-epoch rescue. Total: **600 HPO epoch-units** per newly optimized direction (14 low-fidelity and 8 full-fidelity evaluations).
- **Validation:** Hyperparameters are selected on source-year validation mAP50–95 only; opposite-year test images remain isolated until selection and final training are complete.
- **Final evaluation:** Retrain the frozen configuration for 40 epochs with seeds **42, 123, and 999**, then report mean ± SD on the target-year test split.
- **2021 → 2022:** Stratified subsampling of the source-year training split to match Year2022's training-set size; always optimize and retrain afresh.
- **2022 → 2021:** Reuse an existing **compatible** 600-unit Year2022 single-year run (including three final checkpoints) if rigorously verified; otherwise optimize and retrain afresh. Reused runs consume zero **new** training epoch-units, but retain their historical 600-unit HPO budget.

**Repository layout:** Open Jupyter from the repository root. Place datasets in `Datasets/Year2021/` and `Datasets/Year2022/`, or set `DATASET_PATH_OVERRIDES` in Module 1. Optionally provide an existing single-year Year2022 AF-MCSO run using `YEAR2022_SINGLE_YEAR_AF_MCSO_RUN_OVERRIDE`. Outputs are written to `outputs/`.

**Execution safety:** `RUN_CROSS_YEAR_AF_MCSO = False` by default (final controller cell). Set it to `True` only when both source-year datasets and training environment are ready. No training is performed by merely opening the notebook.

## Module 1: Portable cross-year configuration, checkpoint reuse, and the AF-MCSO-600 protocol

The default workspace is the current repository directory (`Path.cwd()`). Set explicit dataset/checkpoint overrides only if those files are elsewhere. The optimizer remains unchanged from the newer 600-epoch implementation.

In [ ]:
from pathlib import Path
from datetime import datetime
from statistics import mean, stdev
from typing import Dict, Any, List, Optional, Tuple
from collections import defaultdict, Counter

import csv
import json
import math
import os
import random
import re
import shutil
import sys
import time
import uuid
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import torch
import ultralytics
from ultralytics import YOLO

# ============================================================
# A. Both cross-year directions and optional overrides
# ============================================================
# Launch the notebook from your GitHub repository root, or edit this path.
MAIN_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = MAIN_DIRECTORY
WORKSPACE_ROOT = REPO_ROOT
DATASETS_ROOT = REPO_ROOT / "Datasets"
DATASET_YEARS = ("2021", "2022")
DIRECTIONS = (("2021", "2022"), ("2022", "2021"))
DATASET_PATH_OVERRIDES = {"2021": None, "2022": None}
DATASET_SOURCE_BY_YEAR: Dict[str, Path] = {}

# Optional override: path to a completed AF-MCSO single-year Year2022 run,
# e.g. outputs/YOLOv8m_SingleYear_AF_MCSO_Y2022/runs/af_mcso20to40_600/20261115_140000
YEAR2022_SINGLE_YEAR_AF_MCSO_RUN_OVERRIDE = None
# Optional override for the retained 2022 split manifest directory:
# outputs/YOLOv8m_SingleYear_AF_MCSO_Y2022/splits/Year2022
YEAR2022_SINGLE_YEAR_SPLITS_OVERRIDE = None
# True -> ignore prior single-year weights and always train the reverse direction afresh.
FORCE_FRESH_2022_TO_2021 = False

# Match the attached DEHB cross-year protocol only when 2021 is SOURCE.
SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE = True
SUBSAMPLE_TARGET_MODE = "target_train_size"
SUBSAMPLE_SEED_OFFSET = 10000

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
COMBINED_OUTPUT_ROOT = WORKSPACE_ROOT / "outputs" / "YOLOv8m_CrossYear_AF_MCSO_Auto600" / RUN_TIMESTAMP
YEAR = DATASET_YEARS[0]
TRAIN_YEAR, TEST_YEAR = DIRECTIONS[0]
DATASET_TAG = f"Year{TRAIN_YEAR}"
TRAIN_DATASET_TAG = f"Year{TRAIN_YEAR}"
TEST_DATASET_TAG = f"Year{TEST_YEAR}"
CROSS_YEAR_TAG = f"Y{TRAIN_YEAR}_to_Y{TEST_YEAR}"
SOURCE_DATASET_ROOT = DATASETS_ROOT / TRAIN_DATASET_TAG
TRAIN_SOURCE_ROOT = SOURCE_DATASET_ROOT
TEST_TARGET_ROOT = DATASETS_ROOT / TEST_DATASET_TAG

# ============================================================
# B. Atomically switch all per-direction paths
# ============================================================
def activate_cross_year_direction(source_year: str, target_year: str) -> None:
    global TRAIN_YEAR, TEST_YEAR, YEAR, DATASET_TAG, TRAIN_DATASET_TAG, TEST_DATASET_TAG
    global CROSS_YEAR_TAG, SOURCE_DATASET_ROOT, TRAIN_SOURCE_ROOT, TEST_TARGET_ROOT
    global PROJECT_ROOT, CONFIGS_ROOT, RUNS_ROOT, SPLITS_ROOT, STATS_ROOT, YOLO_DATA_ROOT
    global RUN_ROOT, HPO_RUNS_ROOT, FINAL_TRAINING_ROOT, TEST_EVAL_ROOT
    global RUN_STATS_ROOT, RUN_PLOTS_ROOT
    source_year, target_year = str(source_year), str(target_year)
    if (source_year, target_year) not in DIRECTIONS:
        raise ValueError(f"Invalid cross-year direction: {source_year}->{target_year}")
    TRAIN_YEAR, TEST_YEAR = source_year, target_year
    YEAR = source_year  # original AF-MCSO trainer labels training runs by source year
    DATASET_TAG = TRAIN_DATASET_TAG = f"Year{source_year}"
    TEST_DATASET_TAG = f"Year{target_year}"
    CROSS_YEAR_TAG = f"Y{source_year}_to_Y{target_year}"
    SOURCE_DATASET_ROOT = TRAIN_SOURCE_ROOT = DATASET_SOURCE_BY_YEAR.get(
        source_year, DATASETS_ROOT / TRAIN_DATASET_TAG
    )
    TEST_TARGET_ROOT = DATASET_SOURCE_BY_YEAR.get(target_year, DATASETS_ROOT / TEST_DATASET_TAG)
    PROJECT_ROOT = REPO_ROOT / "outputs" / f"YOLOv8m_CrossYear_AF_MCSO_{CROSS_YEAR_TAG}"
    CONFIGS_ROOT = PROJECT_ROOT / "configs"
    RUNS_ROOT = PROJECT_ROOT / "runs"
    SPLITS_ROOT = PROJECT_ROOT / "splits"
    STATS_ROOT = PROJECT_ROOT / "stats"
    YOLO_DATA_ROOT = PROJECT_ROOT / "yolo_data"
    RUN_ROOT = RUNS_ROOT / "af_mcso20to40_600" / RUN_TIMESTAMP
    HPO_RUNS_ROOT = RUN_ROOT / "hpo"
    FINAL_TRAINING_ROOT = RUN_ROOT / "final40"
    TEST_EVAL_ROOT = RUN_ROOT / "target_test"
    RUN_STATS_ROOT = RUN_ROOT / "stats"
    RUN_PLOTS_ROOT = RUN_ROOT / "plots"
    print(f"Activated {CROSS_YEAR_TAG} | source={SOURCE_DATASET_ROOT} | target={TEST_TARGET_ROOT}")

activate_cross_year_direction(*DIRECTIONS[0])

def ensure_output_directories() -> None:
    for folder in (PROJECT_ROOT, CONFIGS_ROOT, RUNS_ROOT, SPLITS_ROOT, STATS_ROOT,
                   YOLO_DATA_ROOT, RUN_ROOT, HPO_RUNS_ROOT, FINAL_TRAINING_ROOT,
                   TEST_EVAL_ROOT, RUN_STATS_ROOT, RUN_PLOTS_ROOT):
        folder.mkdir(parents=True, exist_ok=True)

def portable_path(path: Path) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(REPO_ROOT.resolve()).as_posix()
    except (ValueError, OSError):
        return str(path.name)

def resolve_repo_path(path_value: str) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else REPO_ROOT / path

def sanitize_exception_message(error: Exception) -> str:
    message = f"{type(error).__name__}: {error}"
    for value in (str(REPO_ROOT.resolve()), str(Path.home().resolve())):
        if value:
            message = message.replace(value, "<local_path>")
            message = message.replace(value.replace("\\", "/"), "<local_path>")
    return message

# ============================================================
# C. Target classes and aliases
# ============================================================
CLASS_NAMES = [
    "Waterhemp",
    "Carpetweed",
    "Morningglory",
    "Goosegrass",
    "SpottedSpurge",
    "PalmerAmaranth",
    "Purslane",
    "Ragweed",
]
TARGET_CLASS_SET = set(CLASS_NAMES)

LABEL_ALIASES = {
    "morninglory": "Morningglory",
    "morning glory": "Morningglory",
    "morningglory": "Morningglory",
    "spotted spurge": "SpottedSpurge",
    "spottedspurge": "SpottedSpurge",
    "palmer amaranth": "PalmerAmaranth",
    "palmeramaranth": "PalmerAmaranth",
}


# ============================================================
# D. Split and seed protocol
# ============================================================
TRAIN_RATIO = 0.70
VALID_RATIO = 0.15
TEST_RATIO = 0.15

OPTIMIZATION_SPLIT_SEED = 42
OPTIMIZATION_TRAIN_SEED = 42
FINAL_EVALUATION_SEEDS = [42, 123, 999]

MCSO_RANDOM_SEED = 9187


# ============================================================
# E. Shared YOLO runtime protocol
# ============================================================
MODEL_NAME = "yolov8m.pt"
IMAGE_SIZE = 640
BATCH_SIZE = 16
WORKERS = 8
PATIENCE = 0
IMAGE_PLACEMENT_MODE = "symlink"

LOW_FIDELITY_EPOCHS = 20
LOW_FIDELITY_WARMUP_EPOCHS = 1.5
LOW_FIDELITY_CLOSE_MOSAIC = 5

FULL_FIDELITY_EPOCHS = 40
FULL_FIDELITY_WARMUP_EPOCHS = 3.0
FULL_FIDELITY_CLOSE_MOSAIC = 10

FINAL_EPOCHS = 40
FINAL_WARMUP_EPOCHS = 3.0
FINAL_CLOSE_MOSAIC = 10


# ============================================================
# F. Frozen focused 5D search space
# ============================================================
SEARCH_SPACE_VERSION = "Focused5D_AFMCSO20to40_600_mAP5095_v2"

SEARCH_SPACE = {
    "learning_rate": {"low": 0.0010, "high": 0.0100, "distribution": "log"},
    "lrf":           {"low": 0.0100, "high": 0.2000, "distribution": "log"},
    "momentum":      {"low": 0.8500, "high": 0.9700, "distribution": "linear"},
    "weight_decay":  {"low": 0.0001, "high": 0.0010, "distribution": "log"},
    "scale":         {"low": 0.2000, "high": 0.8000, "distribution": "linear"},
}

SEARCH_PARAMETER_ORDER = [
    "learning_rate",
    "lrf",
    "momentum",
    "weight_decay",
    "scale",
]

REFERENCE_UNTUNED_VALUES = {
    "learning_rate": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "scale": 0.50,
}

for parameter_name, reference_value in REFERENCE_UNTUNED_VALUES.items():
    spec = SEARCH_SPACE[parameter_name]
    if not float(spec["low"]) <= float(reference_value) <= float(spec["high"]):
        raise RuntimeError(
            f"Reference value for {parameter_name}={reference_value} is outside "
            f"the search range [{spec['low']}, {spec['high']}]."
        )


# ============================================================
# G. Fixed non-tuned training settings
# ============================================================
FIXED_TRAINING_ARGS = {
    "optimizer": "SGD",
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "nbs": 64,
    "mosaic": 1.0,
    "hsv_h": 0.015,
    "hsv_s": 0.70,
    "hsv_v": 0.40,
    "degrees": 0.0,
    "translate": 0.10,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mixup": 0.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "cache": "disk",
    "cos_lr": False,
    "rect": False,
}


# ============================================================
# H. AF-MCSO search mechanics
# ============================================================
INITIAL_NESTS = 4
INITIAL_CONFIRMATIONS = 2
INITIAL_MAXIMIN_POOL = 1000

MCSO_GENERATIONS = 5
CANDIDATES_PER_GENERATION = 2
PROMOTIONS_PER_GENERATION = 1

ELITE_ARCHIVE_SIZE = 4
ARCHIVE_MIN_DISTANCE = 0.08

LEVY_BETA = 1.5
# Broad early movement, progressively finer local exploitation in later generations.
GENERATION_STEP_SCALES = [0.18, 0.14, 0.10, 0.07, 0.04]
# One additional 40-epoch confirmation selected across previously skipped screens.
FINAL_RESCUE_PROMOTIONS = 1
FINAL_RESCUE_NOVELTY_WEIGHT = 0.01
STAGNATION_TOLERANCE = 1e-4
STAGNATION_SCALE_MULTIPLIER = 1.75
OPPOSITION_DIRECTION_WEIGHT = 0.50
STAGNATION_OPPOSITION_WEIGHT = 0.70
IMMIGRANT_POOL_SIZE = 500
SIGNATURE_PRECISION = 12

AF_K_NEIGHBORS = 4
AF_MIN_HISTORY_PAIRS = 2
AF_DISTANCE_EPS = 1e-12


# ============================================================
# I. Exact per-year 600-unit HPO budget
# ============================================================
# Stage 0: 4 x 20 + 2 x 40 = 160
# Generations 1-5: 5 x (2 x 20 + 1 x 40) = 400
# Final archive rescue: 1 x 40 = 40
# Total HPO: 600 epoch-units per year.
INITIAL_STAGE_UNITS = (
    INITIAL_NESTS * LOW_FIDELITY_EPOCHS
    + INITIAL_CONFIRMATIONS * FULL_FIDELITY_EPOCHS
)
PER_GENERATION_UNITS = (
    CANDIDATES_PER_GENERATION * LOW_FIDELITY_EPOCHS
    + PROMOTIONS_PER_GENERATION * FULL_FIDELITY_EPOCHS
)
FINAL_RESCUE_UNITS = FINAL_RESCUE_PROMOTIONS * FULL_FIDELITY_EPOCHS
PLANNED_HPO_EPOCH_UNITS = (
    INITIAL_STAGE_UNITS
    + MCSO_GENERATIONS * PER_GENERATION_UNITS
    + FINAL_RESCUE_UNITS
)

EXPECTED_LOW_FIDELITY_EVALUATIONS = (
    INITIAL_NESTS
    + MCSO_GENERATIONS * CANDIDATES_PER_GENERATION
)
EXPECTED_FULL_FIDELITY_EVALUATIONS = (
    INITIAL_CONFIRMATIONS
    + MCSO_GENERATIONS * PROMOTIONS_PER_GENERATION
    + FINAL_RESCUE_PROMOTIONS
)
EXPECTED_TOTAL_HPO_EVALUATIONS = (
    EXPECTED_LOW_FIDELITY_EVALUATIONS
    + EXPECTED_FULL_FIDELITY_EVALUATIONS
)

FINAL_EVALUATION_EPOCH_UNITS = len(FINAL_EVALUATION_SEEDS) * FINAL_EPOCHS
PLANNED_FULL_WORKFLOW_EPOCH_UNITS = (
    PLANNED_HPO_EPOCH_UNITS + FINAL_EVALUATION_EPOCH_UNITS
)

assert INITIAL_STAGE_UNITS == 160
assert PER_GENERATION_UNITS == 80
assert FINAL_RESCUE_UNITS == 40
assert PLANNED_HPO_EPOCH_UNITS == 600
assert EXPECTED_LOW_FIDELITY_EVALUATIONS == 14
assert EXPECTED_FULL_FIDELITY_EVALUATIONS == 8
assert EXPECTED_TOTAL_HPO_EVALUATIONS == 22
assert FINAL_EVALUATION_EPOCH_UNITS == 120
assert PLANNED_FULL_WORKFLOW_EPOCH_UNITS == 720
assert len(GENERATION_STEP_SCALES) == MCSO_GENERATIONS

REQUIRE_FULL_HPO_BUDGET = True
REQUIRE_ALL_FINAL_RUNS = True

STRICT_FITNESS_ALIGNMENT_CHECK = True
FITNESS_ALIGNMENT_TOLERANCE = 0.002


def get_fidelity_controls(budget_epochs: int) -> Tuple[float, int]:
    """Return the frozen 20- or 40-epoch schedule."""
    budget_epochs = int(budget_epochs)
    if budget_epochs == LOW_FIDELITY_EPOCHS:
        return LOW_FIDELITY_WARMUP_EPOCHS, LOW_FIDELITY_CLOSE_MOSAIC
    if budget_epochs == FULL_FIDELITY_EPOCHS:
        return FULL_FIDELITY_WARMUP_EPOCHS, FULL_FIDELITY_CLOSE_MOSAIC
    raise ValueError(f"Unsupported AF-MCSO fidelity: {budget_epochs}")



print("=" * 95)
print("YOLOv8m AF-MCSO CROSS-YEAR: AUTO REUSE OR TRAIN")
print("=" * 95)
print("Directions:", DIRECTIONS)
print("Activated:", CROSS_YEAR_TAG)
print("AF-MCSO objective: source validation mAP50-95 (target test excluded)")
print("Search:", SEARCH_SPACE_VERSION)
print("Schedule: 4x20 + 2x40 + 5x(2x20 + 1x40) + 1x40")
print("HPO units per fresh direction:", PLANNED_HPO_EPOCH_UNITS)
print("Evaluations:", EXPECTED_LOW_FIDELITY_EVALUATIONS, "@20 +",
      EXPECTED_FULL_FIDELITY_EVALUATIONS, "@40")
print("Final seeds:", FINAL_EVALUATION_SEEDS, "(120 extra units if fresh)")
print("Year2021 source train balanced to Year2022 source/target training split:",
      SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE)
print("CUDA available:", torch.cuda.is_available())
print("=" * 95)


## Module 1A: Robust source-year discovery before GPU work

Uses the original AF-MCSO-600 notebook's automatic Year2021 / Year2022 folder discovery and source annotation checks. The controller resolves both years before training or checkpoint use.


In [ ]:
def _normalized_folder_name(name: str) -> str:
    """Normalize folder spellings without broad fuzzy matching."""
    return re.sub(r"[^a-z0-9]", "", str(name).casefold())


def _first_file_matching(root: Path, extensions: set):
    """Lightweight detection: stop searching after the first matching file."""
    try:
        for path in root.rglob("*"):
            if path.is_file() and path.suffix.lower() in extensions:
                return path
    except (OSError, PermissionError):
        return None
    return None


def _dataset_content_status(root: Path) -> Tuple[bool, str]:
    """Verify a source year has images and supported source annotations."""
    if not root.is_dir():
        return False, "folder does not exist"
    image_root = next(
        (candidate for candidate in (
            root / "JPEGImages", root / "images", root
        ) if candidate.is_dir()),
        None,
    )
    if image_root is None:
        return False, "no image directory"
    image_file = _first_file_matching(
        image_root, {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
    )
    if image_file is None:
        return False, "no image files found"
    annotation_file = _first_file_matching(root, {".xml", ".json"})
    if annotation_file is None:
        return False, "no XML/JSON annotation files found"
    return True, "image files and source annotations found"


def _safe_subdirectories(root: Path) -> List[Path]:
    if not root.is_dir():
        return []
    try:
        return sorted(
            (p for p in root.iterdir() if p.is_dir()),
            key=lambda p: p.name.casefold(),
        )
    except (OSError, PermissionError):
        return []


def _candidate_parents() -> List[Path]:
    """Search only source-like folders, never generated YOLO outputs."""
    bases = [WORKSPACE_ROOT, Path.cwd()]
    parents = []
    seen = set()
    source_parent_names = {
        "datasets", "dataset", "data", "raw", "rawdata",
        "2seasonweeddet8", "2seasonweeddet", "weeddata",
    }
    for base in bases:
        for candidate in (
            base / "Datasets", base / "datasets", base / "Dataset",
            base / "dataset", base / "data", base / "Data",
            base / "raw", base / "2SeasonWeedDet8", base,
        ):
            normalized = str(candidate).casefold()
            if normalized not in seen:
                seen.add(normalized)
                parents.append(candidate)
            # One additional level for a downloaded dataset wrapper folder.
            for child in _safe_subdirectories(candidate):
                if _normalized_folder_name(child.name) in source_parent_names:
                    key = str(child).casefold()
                    if key not in seen:
                        seen.add(key)
                        parents.append(child)
    return parents


def _possible_year_folders(year: str) -> Tuple[List[Path], List[Path]]:
    """Return plausible source roots and the parent locations inspected."""
    aliases = {str(year), f"year{year}"}
    parents = _candidate_parents()
    candidates = []
    seen = set()
    for parent in parents:
        for child in _safe_subdirectories(parent):
            if _normalized_folder_name(child.name) not in aliases:
                continue
            # Also allow a downloaded archive with a redundant inner year folder.
            for folder in [
                inner for inner in _safe_subdirectories(child)
                if _normalized_folder_name(inner.name) in aliases
            ] + [child]:
                key = str(folder).casefold()
                if key not in seen:
                    seen.add(key)
                    candidates.append(folder)
    return candidates, parents


def resolve_source_years() -> Dict[str, Path]:
    """Fail clearly before ANY training if a required year is unavailable."""
    resolved = {}
    errors = []
    for year in DATASET_YEARS:
        print(f"\nChecking Year{year} source data ...")
        override = DATASET_PATH_OVERRIDES.get(year)
        if override:
            folder = Path(override).expanduser()
            ok, reason = _dataset_content_status(folder)
            if not ok:
                errors.append(
                    f"Year{year}: override {folder} is invalid ({reason})."
                )
                continue
            print(f"  Found via explicit override: {folder}")
            resolved[year] = folder
            continue

        candidates, parents = _possible_year_folders(year)
        rejected = []
        for candidate in candidates:
            ok, reason = _dataset_content_status(candidate)
            if ok:
                resolved[year] = candidate
                print(f"  Found: {candidate}")
                break
            rejected.append(f"  - {candidate} ({reason})")

        if year not in resolved:
            inspected = [p for p in parents if p.is_dir()]
            nearby = []
            for parent in inspected[:8]:
                names = [p.name for p in _safe_subdirectories(parent)]
                if names:
                    nearby.append(
                        f"  {parent}: {', '.join(names[:18])}"
                    )
            details = (
                f"Year{year}: no usable source folder found.\n"
                f"  Expected example: {MAIN_DIRECTORY / 'Datasets' / ('Year' + year)}\n"
                + ("Recognized names but invalid contents:\n" + "\n".join(rejected) + "\n" if rejected else "")
                + ("Nearby folders:\n" + "\n".join(nearby) + "\n" if nearby else "")
                + f"Set DATASET_PATH_OVERRIDES[{year!r}] to the actual extracted year folder."
            )
            errors.append(details)

    if errors:
        raise FileNotFoundError(
            "Dataset preflight failed; no GPU training was started.\n\n"
            + "\n\n".join(errors)
            + "\n\nNote: This notebook needs all requested years for auto-switching. "
              "Both year folders are required for the automatic two-direction workflow. "
              "Extract any archived datasets or provide DATASET_PATH_OVERRIDES in Module 1."
        )

    print("\nDataset preflight passed. All requested years are available.")
    for year, path in resolved.items():
        print(f"  Year{year}: {path}")
    return resolved


## Module 2: Annotation parsing and normalized record helpers

Reads Pascal VOC XML and supported JSON/VIA annotations, normalizes known class aliases, validates bounding boxes, and prepares records for the eight retained weed classes.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def normalize_label(raw_label: str, label_aliases: Dict[str, str]) -> str:
    """Normalize one class label using the supplied alias map."""
    cleaned = str(raw_label).strip()
    return label_aliases.get(cleaned.lower(), cleaned)


def read_image_size(image_path: Path) -> Tuple[int, int]:
    """Return image (width, height), failing clearly for unreadable files."""
    try:
        with Image.open(image_path) as image:
            return image.size
    except (UnidentifiedImageError, OSError) as error:
        raise ValueError(f"Unreadable image: {image_path}\n{error}") from error


def parse_pascal_voc_xml(
    xml_path: Path,
    label_aliases: Dict[str, str],
) -> Tuple[Optional[Tuple[int, int]], List[Dict[str, Any]]]:
    """Parse one Pascal VOC XML file."""
    root = ET.parse(xml_path).getroot()
    xml_size = None
    size_node = root.find("size")

    if size_node is not None:
        width_text = size_node.findtext("width")
        height_text = size_node.findtext("height")
        if width_text and height_text:
            xml_size = (int(float(width_text)), int(float(height_text)))

    objects: List[Dict[str, Any]] = []

    for object_node in root.findall("object"):
        raw_name = object_node.findtext("name")
        box_node = object_node.find("bndbox")
        if not raw_name or box_node is None:
            continue

        coordinates = [
            box_node.findtext("xmin"),
            box_node.findtext("ymin"),
            box_node.findtext("xmax"),
            box_node.findtext("ymax"),
        ]
        if any(value is None for value in coordinates):
            continue

        try:
            x1, y1, x2, y2 = [float(value) for value in coordinates]
        except (TypeError, ValueError):
            continue

        if x2 <= x1 + 1 or y2 <= y1 + 1:
            continue

        objects.append({
            "label": normalize_label(raw_name, label_aliases),
            "bbox_xyxy": [x1, y1, x2, y2],
        })

    return xml_size, objects


def extract_via_label(
    attributes: Dict[str, Any],
    label_aliases: Dict[str, str],
) -> Optional[str]:
    """Extract a label from common VIA region-attribute layouts."""
    preferred_keys = [
        "label", "class", "class_name", "name", "species", "type", "weed", "category"
    ]

    for key in preferred_keys:
        value = attributes.get(key)
        if isinstance(value, str) and value.strip():
            return normalize_label(value, label_aliases)
        if isinstance(value, dict):
            for nested_key, selected in value.items():
                if selected in {True, 1, "1", "true", "True", "yes", "Yes"}:
                    return normalize_label(nested_key, label_aliases)
            if len(value) == 1:
                return normalize_label(next(iter(value.keys())), label_aliases)

    for value in attributes.values():
        if isinstance(value, str) and value.strip():
            return normalize_label(value, label_aliases)

    return None


def parse_json_annotation(
    json_path: Path,
    label_aliases: Dict[str, str],
) -> List[Dict[str, Any]]:
    """Parse LabelMe-style or VIA-style JSON annotations."""
    data = json.loads(json_path.read_text(encoding="utf-8"))
    objects: List[Dict[str, Any]] = []

    # LabelMe
    if isinstance(data, dict) and isinstance(data.get("shapes"), list):
        for shape in data["shapes"]:
            label = normalize_label(shape.get("label", ""), label_aliases)
            points = shape.get("points", [])
            if not label or len(points) < 2:
                continue

            xs = [float(point[0]) for point in points]
            ys = [float(point[1]) for point in points]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)

            if x2 > x1 + 1 and y2 > y1 + 1:
                objects.append({"label": label, "bbox_xyxy": [x1, y1, x2, y2]})
        return objects

    # VIA
    metadata = data.get("_via_img_metadata", data) if isinstance(data, dict) else {}
    if not isinstance(metadata, dict):
        return objects

    for entry in metadata.values():
        if not isinstance(entry, dict):
            continue
        regions = entry.get("regions", [])
        if isinstance(regions, dict):
            regions = [regions] if "shape_attributes" in regions else list(regions.values())
        if not isinstance(regions, list):
            continue

        for region in regions:
            shape = region.get("shape_attributes", {}) or {}
            attributes = region.get("region_attributes", {}) or {}
            if shape.get("name") != "rect":
                continue

            values = [shape.get("x"), shape.get("y"), shape.get("width"), shape.get("height")]
            if any(value is None for value in values):
                continue

            label = extract_via_label(attributes, label_aliases)
            if not label:
                continue

            try:
                x, y, width, height = [float(value) for value in values]
            except (TypeError, ValueError):
                continue

            if width > 1 and height > 1:
                objects.append({
                    "label": label,
                    "bbox_xyxy": [x, y, x + width, y + height],
                })

    return objects


def clip_box(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Optional[List[float]]:
    """Clip one xyxy box to image boundaries."""
    x1, y1, x2, y2 = box
    x1 = max(0.0, min(float(x1), image_width - 1.0))
    y1 = max(0.0, min(float(y1), image_height - 1.0))
    x2 = max(0.0, min(float(x2), float(image_width)))
    y2 = max(0.0, min(float(y2), float(image_height)))

    if x2 <= x1 + 1 or y2 <= y1 + 1:
        return None
    return [x1, y1, x2, y2]

## Module 3: Source/target record parsing and class audits

Preserves the original AF-MCSO annotation readers and eight-class order. Both years are scanned independently, and source/target records are kept separate.


In [ ]:
def build_clean_records(
    source_root: Path,
    target_class_set: set,
    label_aliases: Dict[str, str],
) -> Tuple[List[Dict[str, Any]], Dict[str, int]]:
    """Build clean records from the source dataset."""
    source_root = Path(source_root)
    if not source_root.exists():
        raise FileNotFoundError(f"Source dataset not found: {source_root.resolve()}")

    image_root = next(
        (p for p in [source_root / "JPEGImages", source_root / "images", source_root] if p.exists()),
        None,
    )
    if image_root is None:
        raise FileNotFoundError(f"No image root found under {source_root.resolve()}")

    image_paths = sorted(
        p for p in image_root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not image_paths:
        raise FileNotFoundError(f"No images found below: {image_root.resolve()}")

    image_names = [p.name for p in image_paths]
    if len(image_names) != len(set(image_names)):
        raise ValueError("Duplicate image filenames found; flat YOLO folders require unique filenames.")

    xml_by_stem = {p.stem: p for p in source_root.rglob("*.xml")}
    json_by_stem = {p.stem: p for p in source_root.rglob("*.json")}

    records: List[Dict[str, Any]] = []
    stats = {
        "images_found": len(image_paths),
        "records_kept": 0,
        "missing_annotation": 0,
        "empty_after_filtering": 0,
        "unreadable_images": 0,
        "invalid_boxes_removed": 0,
        "non_target_objects_removed": 0,
        "used_xml": 0,
        "used_json": 0,
    }

    for image_path in image_paths:
        xml_path = xml_by_stem.get(image_path.stem)
        json_path = json_by_stem.get(image_path.stem)
        annotation_size = None
        objects: List[Dict[str, Any]] = []
        source_used = None

        try:
            # Same preference as the vanilla pipeline: XML first, JSON fallback.
            if xml_path is not None:
                annotation_size, xml_objects = parse_pascal_voc_xml(xml_path, label_aliases)
                if xml_objects:
                    objects = xml_objects
                    source_used = "xml"

            if not objects and json_path is not None:
                objects = parse_json_annotation(json_path, label_aliases)
                if objects:
                    source_used = "json"
                    annotation_size = None

            if not objects:
                stats["missing_annotation"] += 1
                continue

            if annotation_size is None:
                width, height = read_image_size(image_path)
            else:
                width, height = annotation_size

        except Exception as error:
            stats["unreadable_images"] += 1
            print(f"Skipping {image_path.name}: {error}")
            continue

        clean_objects: List[Dict[str, Any]] = []
        for obj in objects:
            label = normalize_label(obj["label"], label_aliases)
            if label not in target_class_set:
                stats["non_target_objects_removed"] += 1
                continue

            clipped = clip_box(obj["bbox_xyxy"], width, height)
            if clipped is None:
                stats["invalid_boxes_removed"] += 1
                continue

            clean_objects.append({"label": label, "bbox_xyxy": clipped})

        if not clean_objects:
            stats["empty_after_filtering"] += 1
            continue

        stats[f"used_{source_used}"] += 1
        records.append({
            "img_path": str(image_path.resolve()),
            "file_name": image_path.name,
            "width": int(width),
            "height": int(height),
            "objects": clean_objects,
        })

    records.sort(key=lambda row: row["file_name"])
    stats["records_kept"] = len(records)
    if not records:
        raise RuntimeError("No usable records remained after filtering.")

    return records, stats


def audit_records(
    records_to_audit: List[Dict[str, Any]],
    class_names: List[str],
) -> Dict[str, Any]:
    """Count boxes and image occurrences per class."""
    box_counts = {name: 0 for name in class_names}
    image_counts = {name: 0 for name in class_names}

    for record in records_to_audit:
        labels = [obj["label"] for obj in record["objects"]]
        for label in labels:
            box_counts[label] += 1
        for label in set(labels):
            image_counts[label] += 1

    return {
        "num_images": len(records_to_audit),
        "num_boxes": int(sum(box_counts.values())),
        "box_count_per_class": box_counts,
        "image_count_per_class": image_counts,
        "missing_classes": [name for name in class_names if box_counts[name] == 0],
    }

def plot_class_distribution(
    audit: Dict[str, Any],
    class_names: List[str],
    output_root: Path,
) -> None:
    """Save one compact box-count plot for the selected dataset."""
    output_root.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 5))
    plt.bar(class_names, [audit["box_count_per_class"][name] for name in class_names])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Number of boxes")
    plt.title(f"{DATASET_TAG}: box count per class")
    plt.tight_layout()
    plt.savefig(output_root / "dataset_box_count_per_class.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()


def prepare_selected_year_records() -> Tuple[
    List[Dict[str, Any]],
    Dict[str, int],
    Dict[str, Any],
]:
    """Scan, validate and audit the selected year inside the controlled workflow."""
    if not SOURCE_DATASET_ROOT.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {portable_path(SOURCE_DATASET_ROOT)}. "
            "Expected repository layout: Datasets/Year2021 and Datasets/Year2022."
        )

    records, record_statistics = build_clean_records(
        source_root=SOURCE_DATASET_ROOT,
        target_class_set=TARGET_CLASS_SET,
        label_aliases=LABEL_ALIASES,
    )
    full_dataset_audit = audit_records(records, CLASS_NAMES)

    if full_dataset_audit["missing_classes"]:
        raise RuntimeError(
            f"Target classes missing from dataset: {full_dataset_audit['missing_classes']}"
        )

    STATS_ROOT.mkdir(parents=True, exist_ok=True)
    (STATS_ROOT / "dataset_scan_summary.json").write_text(
        json.dumps(record_statistics, indent=2), encoding="utf-8"
    )
    (STATS_ROOT / "dataset_class_audit.json").write_text(
        json.dumps(full_dataset_audit, indent=2), encoding="utf-8"
    )

    print("\nDATASET SCAN")
    print(json.dumps(record_statistics, indent=2))
    print("\nDATASET CLASS AUDIT")
    print(json.dumps(full_dataset_audit, indent=2))

    plot_class_distribution(full_dataset_audit, CLASS_NAMES, RUN_PLOTS_ROOT)
    return records, record_statistics, full_dataset_audit


## Module 4: Independent source/target splits with DEHB-matched 2021 balancing

Use the original AF-MCSO deterministic 70/15/15 splitter on **both years**. Effective train/validation come only from the source year; effective test comes only from the target. When Year2021 is source, stratified downsample its train split to the Year2022 train split size, exactly as in the attached DEHB cross-year notebook.


In [ ]:
def split_records_deterministically(
    all_records: List[Dict[str, Any]],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    seed: int,
) -> Dict[str, List[Dict[str, Any]]]:
    """Create one reproducible train/valid/test split."""
    if abs(train_ratio + valid_ratio + test_ratio - 1.0) > 1e-9:
        raise ValueError("Split ratios must sum to 1.0.")

    if len(all_records) < 3:
        raise ValueError("At least three records are required for a three-way split.")

    file_names = [str(row["file_name"]) for row in all_records]
    if len(file_names) != len(set(file_names)):
        duplicates = sorted(
            name for name in set(file_names) if file_names.count(name) > 1
        )
        raise RuntimeError(
            "Duplicate image filenames would overwrite YOLO export files: "
            + ", ".join(duplicates[:10])
        )

    shuffled = list(all_records)
    random.Random(seed).shuffle(shuffled)

    total = len(shuffled)
    n_train = int(total * train_ratio)
    n_valid = int(total * valid_ratio)

    split_records = {
        "train": shuffled[:n_train],
        "valid": shuffled[n_train:n_train + n_valid],
        "test": shuffled[n_train + n_valid:],
    }

    split_names = {
        split_name: {str(row["file_name"]) for row in rows}
        for split_name, rows in split_records.items()
    }
    overlap_pairs = [
        ("train", "valid"),
        ("train", "test"),
        ("valid", "test"),
    ]
    for left, right in overlap_pairs:
        overlap = split_names[left] & split_names[right]
        if overlap:
            raise RuntimeError(
                f"Data leakage detected between {left} and {right}: "
                f"{sorted(overlap)[:10]}"
            )

    return split_records


def audit_split_coverage(
    split_records: Dict[str, List[Dict[str, Any]]],
    class_names: List[str],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit class coverage for train, valid and test splits."""
    report: Dict[str, Any] = {}
    for split_name, rows in split_records.items():
        split_audit = audit_records(rows, class_names)
        report[split_name] = split_audit
        print(
            f"{split_name:8s} images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )
        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Split '{split_name}' is missing classes: {split_audit['missing_classes']}"
            )
    return report


# DEHB-compatible cross-year source-size balancing and leakage checks.
def get_record_primary_label(record: Dict[str, Any]) -> str:
    """
    Assign one primary class to an image for stratified subsampling.

    For multi-class images, use the most frequent class.
    Ties follow the fixed CLASS_NAMES order.
    """
    labels = [obj["label"] for obj in record.get("objects", [])]
    if not labels:
        return "unknown"

    counts = Counter(labels)
    return max(
        CLASS_NAMES,
        key=lambda class_name: (
            counts.get(class_name, 0),
            -CLASS_NAMES.index(class_name),
        ),
    )


def _exact_stratified_allocation(
    groups: Dict[str, List[Dict[str, Any]]],
    target_size: int,
) -> Dict[str, int]:
    """
    Allocate an exact target size approximately in proportion to group sizes,
    while retaining at least one record per represented group when possible.
    """
    labels = [label for label, group in groups.items() if group]
    total = sum(len(groups[label]) for label in labels)

    if target_size < 0 or target_size > total:
        raise ValueError("target_size must be between 0 and the number of available records.")
    if target_size == 0:
        return {label: 0 for label in labels}

    raw = {label: target_size * len(groups[label]) / total for label in labels}
    allocation = {label: min(int(math.floor(raw[label])), len(groups[label])) for label in labels}

    # Preserve represented classes where the requested sample is large enough.
    if target_size >= len(labels):
        for label in labels:
            if allocation[label] == 0:
                allocation[label] = 1

    # If minimum-one adjustments overshoot, remove from the most over-allocated
    # groups while keeping at least one when that rule is active.
    min_allowed = 1 if target_size >= len(labels) else 0
    while sum(allocation.values()) > target_size:
        reducible = [
            label for label in labels
            if allocation[label] > min_allowed
        ]
        if not reducible:
            raise RuntimeError("Could not reduce stratified allocation to the exact target size.")
        label = max(
            reducible,
            key=lambda name: (allocation[name] - raw[name], allocation[name], len(groups[name])),
        )
        allocation[label] -= 1

    # Fill remaining slots using largest remainder, respecting group capacity.
    while sum(allocation.values()) < target_size:
        candidates = [
            label for label in labels
            if allocation[label] < len(groups[label])
        ]
        if not candidates:
            raise RuntimeError("Could not fill stratified allocation to the exact target size.")
        label = max(
            candidates,
            key=lambda name: (raw[name] - allocation[name], len(groups[name])),
        )
        allocation[label] += 1

    if sum(allocation.values()) != target_size:
        raise RuntimeError("Stratified allocation did not reach the requested target size.")

    return allocation


def stratified_subsample_records(
    records: List[Dict[str, Any]],
    target_size: int,
    seed: int,
) -> List[Dict[str, Any]]:
    """Stratified image-level subsampling using the primary class of each image."""
    if target_size >= len(records):
        return list(records)

    rng = random.Random(seed)
    groups: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    for record in records:
        groups[get_record_primary_label(record)].append(record)

    for group in groups.values():
        rng.shuffle(group)

    allocation = _exact_stratified_allocation(groups, target_size)

    sampled: List[Dict[str, Any]] = []
    for label, group in groups.items():
        sampled.extend(group[:allocation.get(label, 0)])

    rng.shuffle(sampled)

    if len(sampled) != target_size:
        raise RuntimeError(
            f"Subsampling produced {len(sampled)} records; expected exactly {target_size}."
        )
    return sampled


def maybe_subsample_source_train(
    source_train_records: List[Dict[str, Any]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    seed: int,
) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    """Apply the established subsampling rule only when Year2021 is the source year."""
    should_subsample = (
        SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE
        and TRAIN_YEAR == "2021"
    )

    info = {
        "enabled": bool(should_subsample),
        "source_year": TRAIN_YEAR,
        "target_year": TEST_YEAR,
        "mode": SUBSAMPLE_TARGET_MODE,
        "subsample_seed": None,
        "original_source_train_count": len(source_train_records),
        "target_size": None,
        "final_source_train_count": len(source_train_records),
    }

    if not should_subsample:
        return list(source_train_records), info

    if SUBSAMPLE_TARGET_MODE != "target_train_size":
        raise ValueError("This notebook freezes SUBSAMPLE_TARGET_MODE='target_train_size'.")

    target_size = len(target_splits["train"])
    subsample_seed = int(seed + SUBSAMPLE_SEED_OFFSET)

    if target_size > len(source_train_records):
        # Do not upsample. This guard makes the rule explicit and auditable.
        print(
            "WARNING: target train size exceeds Year2021 source train size; "
            "subsampling is skipped because the protocol does not upsample."
        )
        info["target_size"] = int(target_size)
        info["subsample_seed"] = subsample_seed
        return list(source_train_records), info

    sampled = stratified_subsample_records(
        records=source_train_records,
        target_size=target_size,
        seed=subsample_seed,
    )

    info["target_size"] = int(target_size)
    info["subsample_seed"] = subsample_seed
    info["final_source_train_count"] = len(sampled)
    return sampled, info


def _path_set(records: List[Dict[str, Any]]) -> set:
    return {str(Path(record["img_path"]).resolve()) for record in records}


def verify_cross_split_disjointness(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
) -> None:
    """Fail if any physical image path occurs in more than one effective split."""
    train_paths = _path_set(cross_year_splits["train"])
    valid_paths = _path_set(cross_year_splits["valid"])
    test_paths = _path_set(cross_year_splits["test"])

    overlaps = {
        "train_valid": train_paths & valid_paths,
        "train_test": train_paths & test_paths,
        "valid_test": valid_paths & test_paths,
    }
    bad = {name: paths for name, paths in overlaps.items() if paths}
    if bad:
        raise RuntimeError(
            "Cross-year split leakage detected: "
            + ", ".join(f"{name}={len(paths)}" for name, paths in bad.items())
        )


def audit_cross_year_splits(
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    fail_on_missing: bool = True,
) -> Dict[str, Any]:
    """Audit effective train/valid/test class coverage."""
    report: Dict[str, Any] = {}
    for split_name, rows in cross_year_splits.items():
        split_audit = audit_records(rows, CLASS_NAMES)
        report[split_name] = split_audit
        print(
            f"{split_name:8s} images={split_audit['num_images']:5d} "
            f"boxes={split_audit['num_boxes']:6d} "
            f"missing={split_audit['missing_classes']}"
        )
        if fail_on_missing and split_audit["missing_classes"]:
            raise RuntimeError(
                f"Effective split '{split_name}' is missing classes: "
                f"{split_audit['missing_classes']}"
            )
    return report


def make_cross_year_splits_for_seed(
    seed: int,
    source_records: List[Dict[str, Any]],
    target_records: List[Dict[str, Any]],
) -> Tuple[
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    """
    Build source splits, target splits, and the effective cross-year layout.

    Effective YOLO layout:
        train = source train (subsampled only if source is Year2021)
        valid = source validation
        test  = target test
    """
    source_splits = split_records_deterministically(
        source_records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )
    target_splits = split_records_deterministically(
        target_records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed
    )

    effective_source_train, subsample_info = maybe_subsample_source_train(
        source_train_records=source_splits["train"],
        target_splits=target_splits,
        seed=seed,
    )

    cross_year_splits = {
        "train": effective_source_train,
        "valid": source_splits["valid"],
        "test": target_splits["test"],
    }

    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)

    split_info = {
        "seed": int(seed),
        "train_year": TRAIN_YEAR,
        "test_year": TEST_YEAR,
        "source_dataset": TRAIN_DATASET_TAG,
        "target_dataset": TEST_DATASET_TAG,
        "ratios": {
            "train": TRAIN_RATIO,
            "valid": VALID_RATIO,
            "test": TEST_RATIO,
        },
        "source_counts_original": {
            name: len(rows) for name, rows in source_splits.items()
        },
        "target_counts_original": {
            name: len(rows) for name, rows in target_splits.items()
        },
        "effective_cross_year_counts": {
            name: len(rows) for name, rows in cross_year_splits.items()
        },
        "subsample_info": subsample_info,
        "class_coverage": coverage,
    }

    return cross_year_splits, source_splits, target_splits, split_info




## Module 5: Cross-year YOLO export, per-seed split manifests, and integrity checks

Retains the AF-MCSO dataset export helpers. Additionally writes the *full source-year splits*, *full target-year splits*, and *effective train/valid/test* filenames for each seed. Reuse requires the stored single-year Year2022 source split to agree with the current source-year split.


In [ ]:
def remove_file_or_link(path: Path) -> None:
    """Remove a regular file or symlink, including broken symlinks."""
    if path.is_symlink() or path.is_file():
        path.unlink()


def clear_split_directory(directory: Path) -> None:
    """Remove stale files/links from one prepared split directory."""
    directory.mkdir(parents=True, exist_ok=True)
    for path in directory.iterdir():
        if path.is_file() or path.is_symlink():
            path.unlink()


def place_image(source: Path, destination: Path, mode: str) -> str:
    """Place one image using symlink -> hardlink -> copy fallback."""
    if mode not in {"symlink", "hardlink", "copy"}:
        raise ValueError("mode must be 'symlink', 'hardlink', or 'copy'.")

    source = source.resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    remove_file_or_link(destination)

    if mode == "symlink":
        try:
            destination.symlink_to(source)
            return "symlinked"
        except OSError:
            pass

    if mode in {"symlink", "hardlink"}:
        try:
            os.link(source, destination)
            return "hardlinked"
        except OSError:
            pass

    shutil.copy2(source, destination)
    return "copied"


def xyxy_to_yolo(
    box: List[float],
    image_width: int,
    image_height: int,
) -> Tuple[float, float, float, float]:
    """Convert absolute xyxy coordinates to normalized YOLO xywh."""
    x1, y1, x2, y2 = box
    values = (
        ((x1 + x2) / 2.0) / image_width,
        ((y1 + y2) / 2.0) / image_height,
        (x2 - x1) / image_width,
        (y2 - y1) / image_height,
    )
    return tuple(max(0.0, min(1.0, float(value))) for value in values)


def write_yolo_label(
    record: Dict[str, Any],
    output_path: Path,
    class_to_id: Dict[str, int],
) -> None:
    """Write one YOLO label file."""
    lines: List[str] = []
    for obj in record["objects"]:
        class_id = class_to_id[obj["label"]]
        xc, yc, bw, bh = xyxy_to_yolo(
            obj["bbox_xyxy"], record["width"], record["height"]
        )
        lines.append(f"{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def write_dataset_yaml(
    dataset_root: Path,
    class_names: List[str],
) -> Path:
    """Write Ultralytics dataset.yaml."""
    yaml_path = dataset_root / "dataset.yaml"
    lines = [
        f"path: {dataset_root.resolve().as_posix()}",
        "train: images/train",
        "val: images/valid",
        "test: images/test",
        f"nc: {len(class_names)}",
        "names:",
    ]
    lines.extend(f"  {idx}: {name}" for idx, name in enumerate(class_names))
    yaml_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return yaml_path


def count_files(directory: Path, suffix: Optional[str] = None) -> int:
    """Count regular files and symlinks, optionally filtered by suffix."""
    paths = [p for p in directory.iterdir() if p.is_file() or p.is_symlink()]
    if suffix is not None:
        paths = [p for p in paths if p.suffix.lower() == suffix.lower()]
    return len(paths)


def check_yolo_dataset_integrity(dataset_root: Path) -> None:
    """Verify image/label counts for every split."""
    for split_name in ("train", "valid", "test"):
        image_dir = dataset_root / "images" / split_name
        label_dir = dataset_root / "labels" / split_name
        image_count = count_files(image_dir)
        label_count = count_files(label_dir, ".txt")
        print(f"{split_name:8s} images={image_count:5d} | labels={label_count:5d}")
        if image_count != label_count:
            raise RuntimeError(f"Image/label count mismatch in split '{split_name}'.")


def prepare_yolo_dataset(
    all_records: List[Dict[str, Any]],
    output_root: Path,
    manifest_root: Path,
    class_names: List[str],
    train_ratio: float,
    valid_ratio: float,
    test_ratio: float,
    image_mode: str,
    split_seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Build a complete YOLO dataset for one deterministic split seed."""
    output_root = Path(output_root)
    manifest_root = Path(manifest_root)
    class_to_id = {name: idx for idx, name in enumerate(class_names)}

    split_records = split_records_deterministically(
        all_records=all_records,
        train_ratio=train_ratio,
        valid_ratio=valid_ratio,
        test_ratio=test_ratio,
        seed=split_seed,
    )

    coverage_report = audit_split_coverage(
        split_records=split_records,
        class_names=class_names,
        fail_on_missing=True,
    )

    image_dirs = {
        name: output_root / "images" / name for name in ("train", "valid", "test")
    }
    label_dirs = {
        name: output_root / "labels" / name for name in ("train", "valid", "test")
    }

    for directory in [*image_dirs.values(), *label_dirs.values()]:
        clear_split_directory(directory)

    placement_counts = {"symlinked": 0, "hardlinked": 0, "copied": 0}
    manifest = {
        "seed": int(split_seed),
        "ratios": {"train": train_ratio, "valid": valid_ratio, "test": test_ratio},
        "image_mode_requested": image_mode,
        "class_coverage": coverage_report,
        "splits": {},
    }

    manifest_root.mkdir(parents=True, exist_ok=True)

    for split_name, rows in split_records.items():
        filenames: List[str] = []

        for record in rows:
            source = Path(record["img_path"])
            destination = image_dirs[split_name] / record["file_name"]
            action = place_image(source, destination, image_mode)
            placement_counts[action] += 1

            label_path = label_dirs[split_name] / f"{Path(record['file_name']).stem}.txt"
            write_yolo_label(record, label_path, class_to_id)

            filenames.append(record["file_name"])

        (manifest_root / f"{split_name}_filenames.txt").write_text(
            "\n".join(filenames) + "\n", encoding="utf-8"
        )
        manifest["splits"][split_name] = {
            "count": len(rows),
            "filenames": filenames,
        }

    manifest["actual_placement_counts"] = placement_counts
    (manifest_root / "split_manifest.json").write_text(
        json.dumps(manifest, indent=2), encoding="utf-8"
    )

    yaml_path = write_dataset_yaml(output_root, class_names)
    check_yolo_dataset_integrity(output_root)

    print("Dataset YAML:", yaml_path.resolve())
    print("Image placement:", placement_counts)
    return yaml_path, coverage_report


# ============================================================
# Extra cross-year layout: train=source, valid=source, test=target
# ============================================================
def save_cross_year_split_manifest(
    split_records: Dict[str, List[Dict[str, Any]]],
    output_json: Path,
    seed: int,
    dataset_tag: str,
    role: str,
    extra_info: Optional[Dict[str, Any]] = None,
) -> None:
    payload = {
        "dataset_tag": dataset_tag, "role": role, "seed": int(seed),
        "counts": {key: len(rows) for key, rows in split_records.items()},
        "splits": {key: {"count": len(rows),
                         "filenames": [r["file_name"] for r in rows]}
                   for key, rows in split_records.items()},
    }
    if extra_info is not None:
        payload["extra_info"] = extra_info
    save_json(payload, output_json)


def prepare_cross_year_yolo_dataset(
    *,
    cross_year_splits: Dict[str, List[Dict[str, Any]]],
    source_splits: Dict[str, List[Dict[str, Any]]],
    target_splits: Dict[str, List[Dict[str, Any]]],
    split_info: Dict[str, Any],
    output_root: Path,
    manifest_root: Path,
    image_mode: str,
    seed: int,
) -> Tuple[Path, Dict[str, Any]]:
    """Export source train/val + opposite-year test only (never target train/val)."""
    verify_cross_split_disjointness(cross_year_splits)
    coverage = audit_cross_year_splits(cross_year_splits, fail_on_missing=True)
    output_root, manifest_root = Path(output_root), Path(manifest_root)
    image_dirs = {name: output_root / "images" / name for name in ("train", "valid", "test")}
    label_dirs = {name: output_root / "labels" / name for name in ("train", "valid", "test")}
    for directory in (*image_dirs.values(), *label_dirs.values()):
        clear_split_directory(directory)
    class_to_id = {label: idx for idx, label in enumerate(CLASS_NAMES)}
    placement = {"symlinked": 0, "hardlinked": 0, "copied": 0}
    for split_name in ("train", "valid", "test"):
        rows = cross_year_splits[split_name]
        stems = [Path(r["file_name"]).stem for r in rows]
        if len(stems) != len(set(stems)):
            raise RuntimeError(f"Duplicate label stems in cross-year {split_name} split")
        for record in rows:
            action = place_image(Path(record["img_path"]),
                                 image_dirs[split_name] / record["file_name"], image_mode)
            placement[action] += 1
            write_yolo_label(record,
                             label_dirs[split_name] / f"{Path(record['file_name']).stem}.txt",
                             class_to_id)
    yaml_path = write_dataset_yaml(output_root, CLASS_NAMES)
    check_yolo_dataset_integrity(output_root)
    for split_name, rows in cross_year_splits.items():
        if len(list(image_dirs[split_name].iterdir())) != len(rows):
            raise RuntimeError(f"YOLO exported {split_name} count does not match split manifest")
    manifest_root.mkdir(parents=True, exist_ok=True)
    save_cross_year_split_manifest(source_splits,
        manifest_root / f"{TRAIN_DATASET_TAG}_source_seed{seed}.json",
        seed, TRAIN_DATASET_TAG, "full_source_year_splits")
    save_cross_year_split_manifest(target_splits,
        manifest_root / f"{TEST_DATASET_TAG}_target_seed{seed}.json",
        seed, TEST_DATASET_TAG, "full_target_year_splits")
    save_cross_year_split_manifest(cross_year_splits,
        manifest_root / f"cross_{CROSS_YEAR_TAG}_seed{seed}.json",
        seed, CROSS_YEAR_TAG, "effective_cross_year", extra_info=split_info)
    save_json({
        "direction": CROSS_YEAR_TAG, "seed": int(seed), "dataset_yaml": portable_path(yaml_path),
        "class_names": CLASS_NAMES, "image_mode": image_mode,
        "source_root": portable_path(TRAIN_SOURCE_ROOT), "target_root": portable_path(TEST_TARGET_ROOT),
        "counts": {key: len(rows) for key, rows in cross_year_splits.items()},
        "split_info": split_info, "placement": placement,
    }, manifest_root / "yolo_layout_manifest.json")
    return yaml_path, coverage


## Module 6: Original AF-MCSO YOLO training and metrics extraction

The unchanged AF-MCSO helper trains a configuration from `yolov8m.pt` using the source-year train/validation YAML, verifies checkpoint fitness alignment, and returns checkpoint and validation metrics. The cross-year controller prevents target-test exposure to HPO.


In [ ]:
def extract_box_metrics(
    metrics: Any,
    expected_class_names: List[str],
) -> Dict[str, Any]:
    """Extract standard detection metrics; optimization fitness is mAP50-95 only."""
    box = getattr(metrics, "box", None)
    if box is None:
        raise RuntimeError("Ultralytics returned no detection box metrics.")

    precision = float(box.mp)
    recall = float(box.mr)
    map50 = float(box.map50)
    map50_95 = float(box.map)

    names_obj = getattr(metrics, "names", {})
    if isinstance(names_obj, dict):
        class_name_lookup = {int(k): str(v) for k, v in names_obj.items()}
    else:
        class_name_lookup = {idx: str(name) for idx, name in enumerate(names_obj)}

    per_class_rows: List[Dict[str, Any]] = []

    # Prefer represented-class AP + class indices. This avoids silently using
    # box.maps behavior for classes that are absent from a validation split.
    if hasattr(box, "ap") and hasattr(box, "ap_class_index"):
        ap_values = np.asarray(box.ap, dtype=float).reshape(-1)
        class_ids = np.asarray(box.ap_class_index, dtype=int).reshape(-1)

        for class_id, ap_value in zip(class_ids.tolist(), ap_values.tolist()):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })
    else:
        maps = [float(value) for value in box.maps]
        for class_id, ap_value in enumerate(maps):
            per_class_rows.append({
                "class_id": int(class_id),
                "class_name": class_name_lookup.get(int(class_id), str(class_id)),
                "map50_95": float(ap_value),
            })

    represented_ids = {int(row["class_id"]) for row in per_class_rows}
    expected_ids = set(range(len(expected_class_names)))
    missing_ids = sorted(expected_ids - represented_ids)
    if missing_ids:
        missing_names = [expected_class_names[idx] for idx in missing_ids]
        raise RuntimeError(
            "Validation/test metrics are missing represented classes: "
            f"{missing_names}. Check split coverage before trusting per-class metrics."
        )

    sorted_aps = sorted(float(row["map50_95"]) for row in per_class_rows)
    bottom3 = float(mean(sorted_aps[:3])) if len(sorted_aps) >= 3 else float(mean(sorted_aps))

    return {
        "precision": precision,
        "recall": recall,
        "map50": map50,
        "map50_95": map50_95,
        "fitness": map50_95,
        "bottom3_mean_map50_95": bottom3,
        "per_class_map50_95": per_class_rows,
    }


def read_epoch_map50_95_values(results_csv: Path) -> Tuple[str, List[float]]:
    """Read epoch-level mAP50-95 values from an Ultralytics results.csv file."""
    results_csv = Path(results_csv)
    if not results_csv.exists():
        raise FileNotFoundError(f"Training results.csv not found: {results_csv}")

    with results_csv.open("r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        rows = list(reader)
        fieldnames = reader.fieldnames or []

    candidate_columns = [name for name in fieldnames if "mAP50-95" in name]
    if not candidate_columns:
        raise RuntimeError(
            f"Could not find an mAP50-95 column in {results_csv}. "
            f"Available columns: {fieldnames}"
        )

    # Prefer the standard detection metric column if present.
    metric_column = next(
        (name for name in candidate_columns if "metrics/" in name),
        candidate_columns[0],
    )

    values: List[float] = []
    for row in rows:
        raw_value = row.get(metric_column, "")
        try:
            value = float(raw_value)
        except (TypeError, ValueError):
            continue
        if math.isfinite(value):
            values.append(value)

    if not values:
        raise RuntimeError(f"No numeric values found in results column: {metric_column}")

    return metric_column, values


def verify_ultralytics_fitness_alignment(
    results_csv: Path,
    trainer_best_fitness: Optional[float],
    tolerance: float,
    strict: bool,
) -> Dict[str, Any]:
    """Verify that internal best_fitness equals the maximum epoch mAP50-95."""
    metric_column, map95_values = read_epoch_map50_95_values(results_csv)
    max_epoch_map95 = float(max(map95_values))

    if trainer_best_fitness is None or not math.isfinite(float(trainer_best_fitness)):
        report = {
            "status": "UNVERIFIED",
            "metric_column": metric_column,
            "max_epoch_map50_95": max_epoch_map95,
            "trainer_best_fitness": None,
            "absolute_difference": None,
            "tolerance": float(tolerance),
        }
        if strict:
            raise RuntimeError(
                "Could not read Ultralytics trainer.best_fitness, so checkpoint-fitness "
                "alignment could not be verified."
            )
        return report

    trainer_best_fitness = float(trainer_best_fitness)
    difference = abs(trainer_best_fitness - max_epoch_map95)
    status = "PASS" if difference <= tolerance else "FAIL"

    report = {
        "status": status,
        "metric_column": metric_column,
        "max_epoch_map50_95": max_epoch_map95,
        "trainer_best_fitness": trainer_best_fitness,
        "absolute_difference": float(difference),
        "tolerance": float(tolerance),
    }

    if strict and status != "PASS":
        raise RuntimeError(
            "Ultralytics checkpoint fitness does not align with mAP50-95. "
            f"best_fitness={trainer_best_fitness:.6f}, "
            f"max epoch mAP50-95={max_epoch_map95:.6f}, "
            f"difference={difference:.6f}."
        )

    return report


def train_and_validate_configuration(
    *,
    learning_rate: float,
    lrf: float,
    momentum: float,
    weight_decay: float,
    scale: float,
    dataset_yaml: Path,
    project_dir: Path,
    model_name: str,
    epochs: int,
    image_size: int,
    batch_size: int,
    patience: int,
    workers: int,
    seed: int,
    warmup_epochs: float,
    close_mosaic: int,
    fixed_training_args: Dict[str, Any],
    expected_class_names: List[str],
    strict_fitness_alignment_check: bool,
    fitness_alignment_tolerance: float,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Train one configuration from scratch, verify fitness alignment, then validate best.pt."""
    dataset_yaml = Path(dataset_yaml)
    project_dir = Path(project_dir)

    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")
    if learning_rate <= 0 or lrf <= 0:
        raise ValueError("learning_rate and lrf must be greater than 0.")
    if not 0.0 < momentum < 1.0:
        raise ValueError("momentum must be between 0 and 1.")
    if weight_decay < 0:
        raise ValueError("weight_decay cannot be negative.")
    if not 0.0 <= scale <= 1.0:
        raise ValueError("scale must be between 0 and 1.")
    if epochs < 1 or batch_size < 1 or workers < 0:
        raise ValueError("Invalid epochs, batch_size, or workers value.")
    if warmup_epochs < 0 or close_mosaic < 0:
        raise ValueError("warmup_epochs and close_mosaic must be non-negative.")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"
    using_cuda = str(device) != "cpu" and torch.cuda.is_available()

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    trial_id = uuid.uuid4().hex[:8]
    run_name = f"Y{YEAR}_s{seed}_e{epochs}_{trial_id}"
    project_dir.mkdir(parents=True, exist_ok=True)

    hyperparameters = {
        "learning_rate": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "epochs": int(epochs),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
        "seed": int(seed),
        "patience": int(patience),
    }

    print("\n" + "=" * 88)
    print("START TRAINING:", run_name)
    print(json.dumps(hyperparameters, indent=2))
    print("=" * 88)

    train_kwargs = dict(fixed_training_args)
    train_kwargs.update({
        "data": str(dataset_yaml.resolve()),
        "project": str(project_dir.resolve()),
        "name": run_name,
        "epochs": int(epochs),
        "imgsz": int(image_size),
        "batch": int(batch_size),
        "patience": int(patience),
        "workers": int(workers),
        "device": device,
        "seed": int(seed),
        "deterministic": True,
        "pretrained": True,
        "val": True,
        "save": True,
        "save_period": -1,
        "plots": False,
        "verbose": True,
        "amp": bool(using_cuda),
        "lr0": float(learning_rate),
        "lrf": float(lrf),
        "momentum": float(momentum),
        "weight_decay": float(weight_decay),
        "scale": float(scale),
        "warmup_epochs": float(warmup_epochs),
        "close_mosaic": int(close_mosaic),
    })

    model = YOLO(model_name)
    train_start = time.perf_counter()
    train_result = model.train(**train_kwargs)
    training_seconds = time.perf_counter() - train_start

    train_save_dir = getattr(train_result, "save_dir", None)
    if train_save_dir is None and getattr(model, "trainer", None) is not None:
        train_save_dir = getattr(model.trainer, "save_dir", None)
    if train_save_dir is None:
        raise RuntimeError("Training finished but save_dir could not be resolved.")

    train_save_dir = Path(train_save_dir)
    best_checkpoint = train_save_dir / "weights" / "best.pt"
    results_csv = train_save_dir / "results.csv"

    if not best_checkpoint.exists():
        raise FileNotFoundError(f"best.pt not found: {best_checkpoint}")
    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")

    trainer_best_fitness = None
    trainer = getattr(model, "trainer", None)
    if trainer is not None:
        raw_best_fitness = getattr(trainer, "best_fitness", None)
        if raw_best_fitness is not None:
            trainer_best_fitness = float(raw_best_fitness)

    alignment_report = verify_ultralytics_fitness_alignment(
        results_csv=results_csv,
        trainer_best_fitness=trainer_best_fitness,
        tolerance=fitness_alignment_tolerance,
        strict=strict_fitness_alignment_check,
    )

    best_model = YOLO(str(best_checkpoint))
    val_start = time.perf_counter()
    validation_metrics = best_model.val(
        data=str(dataset_yaml.resolve()),
        split="val",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(project_dir.resolve()),
        name=f"{run_name}_val",
        plots=False,
        save_json=False,
        verbose=False,
    )
    validation_seconds = time.perf_counter() - val_start

    metric_data = extract_box_metrics(
        metrics=validation_metrics,
        expected_class_names=expected_class_names,
    )

    peak_gpu_memory_gb = None
    if using_cuda:
        peak_gpu_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

    result = {
        "status": "PASS",
        "trial_id": trial_id,
        "run_name": run_name,
        "hyperparameters": hyperparameters,
        "metrics": metric_data,
        "fitness_alignment": alignment_report,
        "timing": {
            "training_seconds": float(training_seconds),
            "validation_seconds": float(validation_seconds),
            "total_seconds": float(training_seconds + validation_seconds),
        },
        "hardware": {
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0) if using_cuda else None,
            "peak_gpu_memory_gb": (
                float(peak_gpu_memory_gb) if peak_gpu_memory_gb is not None else None
            ),
        },
        "paths": {
            "training_folder": portable_path(train_save_dir),
            "best_checkpoint": portable_path(best_checkpoint),
            "results_csv": portable_path(results_csv),
        },
    }

    (train_save_dir / "trial_summary.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )

    print(
        f"Completed {run_name}: mAP50-95={metric_data['map50_95']:.6f}, "
        f"P={metric_data['precision']:.6f}, R={metric_data['recall']:.6f}, "
        f"alignment={alignment_report['status']}"
    )
    return result

## Module 7: AF-MCSO normalized search, Lévy flights, elite archive, and adaptive fidelity

Unchanged from the supplied AF-MCSO-600 single-year notebook. Direction-specific source data are supplied through the effective cross-year YAML; **no DEHB optimizer mechanics are substituted**.


In [ ]:
def clip01(value: float) -> float:
    return max(0.0, min(1.0, float(value)))


def reflect01(value: float) -> float:
    """Reflect a coordinate into [0,1] without clipping away overshoot information."""
    value = float(value)
    while value < 0.0 or value > 1.0:
        if value < 0.0:
            value = -value
        if value > 1.0:
            value = 2.0 - value
    return clip01(value)


def vector_signature(vector: List[float]) -> Tuple[float, ...]:
    return tuple(round(float(value), SIGNATURE_PRECISION) for value in vector)


def normalized_distance(a: List[float], b: List[float]) -> float:
    return math.sqrt(sum((float(x) - float(y)) ** 2 for x, y in zip(a, b)))


def mean_pairwise_distance(vectors: List[List[float]]) -> float:
    if len(vectors) < 2:
        return 0.0
    distances = []
    for i in range(len(vectors)):
        for j in range(i + 1, len(vectors)):
            distances.append(normalized_distance(vectors[i], vectors[j]))
    return float(mean(distances)) if distances else 0.0


def decode_vector(vector: List[float], config_id: Optional[str] = None) -> Dict[str, Any]:
    """Decode normalized search coordinates into the frozen 5D parameter space."""
    if len(vector) != len(SEARCH_PARAMETER_ORDER):
        raise ValueError("Vector dimensionality does not match SEARCH_PARAMETER_ORDER.")

    config: Dict[str, Any] = {}
    for parameter_name, z in zip(SEARCH_PARAMETER_ORDER, vector):
        z = clip01(z)
        spec = SEARCH_SPACE[parameter_name]
        low = float(spec["low"])
        high = float(spec["high"])

        if spec["distribution"] == "log":
            value = math.exp(math.log(low) + z * (math.log(high) - math.log(low)))
        elif spec["distribution"] == "linear":
            value = low + z * (high - low)
        else:
            raise ValueError(f"Unsupported distribution for {parameter_name}: {spec['distribution']}")

        config[parameter_name] = float(value)

    if config_id is not None:
        config["config_id"] = str(config_id)
    return config


def encode_config(config: Dict[str, Any]) -> List[float]:
    """Encode a 5D configuration into normalized coordinates."""
    vector = []
    for parameter_name in SEARCH_PARAMETER_ORDER:
        spec = SEARCH_SPACE[parameter_name]
        low = float(spec["low"])
        high = float(spec["high"])
        value = float(config[parameter_name])

        if spec["distribution"] == "log":
            z = (math.log(value) - math.log(low)) / (math.log(high) - math.log(low))
        else:
            z = (value - low) / (high - low)
        vector.append(clip01(z))
    return vector


def row_to_vector(row: Dict[str, Any]) -> List[float]:
    return [float(row[f"z{index}"]) for index in range(len(SEARCH_PARAMETER_ORDER))]


def sample_unit_vector(rng: random.Random) -> List[float]:
    return [rng.random() for _ in SEARCH_PARAMETER_ORDER]


def maximin_initial_vectors(
    rng: random.Random,
    n_vectors: int,
    pool_size: int,
) -> List[List[float]]:
    """Choose a diverse initial population from a seeded random candidate pool."""
    if n_vectors < 2:
        raise ValueError("n_vectors must be at least 2.")
    if pool_size < n_vectors:
        raise ValueError("pool_size must be at least n_vectors.")

    pool = [sample_unit_vector(rng) for _ in range(pool_size)]

    # Start with the farthest pair.
    best_pair = None
    best_distance = -1.0
    for i in range(len(pool)):
        for j in range(i + 1, len(pool)):
            distance = normalized_distance(pool[i], pool[j])
            if distance > best_distance:
                best_distance = distance
                best_pair = (i, j)

    if best_pair is None:
        raise RuntimeError("Could not create a maximin initial pair.")

    selected = [pool[best_pair[0]], pool[best_pair[1]]]
    selected_signatures = {vector_signature(vector) for vector in selected}

    while len(selected) < n_vectors:
        best_candidate = None
        best_min_distance = -1.0
        for candidate in pool:
            if vector_signature(candidate) in selected_signatures:
                continue
            min_distance = min(normalized_distance(candidate, existing) for existing in selected)
            if min_distance > best_min_distance:
                best_min_distance = min_distance
                best_candidate = candidate

        if best_candidate is None:
            raise RuntimeError("Maximin initialization could not find enough unique vectors.")

        selected.append(best_candidate)
        selected_signatures.add(vector_signature(best_candidate))

    return selected


def levy_step(rng: random.Random, beta: float = LEVY_BETA) -> float:
    """Mantegna Levy-flight step."""
    sigma_u = (
        (
            math.gamma(1.0 + beta)
            * math.sin(math.pi * beta / 2.0)
        )
        /
        (
            math.gamma((1.0 + beta) / 2.0)
            * beta
            * (2.0 ** ((beta - 1.0) / 2.0))
        )
    ) ** (1.0 / beta)

    u = rng.gauss(0.0, sigma_u)
    v = rng.gauss(0.0, 1.0)
    denominator = max(abs(v) ** (1.0 / beta), 1e-12)
    return max(-5.0, min(5.0, u / denominator))


def local_levy_child(
    parent: List[float],
    rng: random.Random,
    step_scale: float,
) -> List[float]:
    """Exploit around the current full-fidelity elite."""
    child = list(parent)
    coordinate_count = rng.randint(1, min(2, len(child)))
    indices = rng.sample(range(len(child)), k=coordinate_count)
    for index in indices:
        child[index] = reflect01(child[index] + step_scale * levy_step(rng))
    return child


def opposition_levy_child(
    parent: List[float],
    rng: random.Random,
    step_scale: float,
    direction_weight: float,
) -> List[float]:
    """Explore toward the opposite region with a small Levy perturbation."""
    child = list(parent)
    coordinate_count = rng.randint(2, min(4, len(child)))
    indices = rng.sample(range(len(child)), k=coordinate_count)

    for index in indices:
        opposite = 1.0 - parent[index]
        directed = parent[index] + direction_weight * (opposite - parent[index])
        child[index] = reflect01(directed + 0.5 * step_scale * levy_step(rng))
    return child


def elite_recombination_child(
    best_parent: List[float],
    diverse_parent: List[float],
    rng: random.Random,
    step_scale: float,
) -> List[float]:
    """Late-phase exploitation: mix two observed 40-epoch elites, then refine.

    These parents come ONLY from completed full-fidelity trials in the current year.
    """
    child = list(best_parent)
    indices = rng.sample(range(len(child)), k=rng.randint(2, min(4, len(child))))
    for index in indices:
        # Prefer the strongest confirmed parent but borrow information from a
        # diverse confirmed elite; use a small stable local Levy adjustment.
        recombined = 0.7 * best_parent[index] + 0.3 * diverse_parent[index]
        child[index] = reflect01(recombined + 0.35 * step_scale * levy_step(rng))
    return child


def maximin_immigrant(
    existing_vectors: List[List[float]],
    rng: random.Random,
    pool_size: int = IMMIGRANT_POOL_SIZE,
) -> List[float]:
    """Choose a global candidate far from the already evaluated region."""
    pool = [sample_unit_vector(rng) for _ in range(pool_size)]
    if not existing_vectors:
        return pool[0]

    return max(
        pool,
        key=lambda candidate: min(
            normalized_distance(candidate, existing)
            for existing in existing_vectors
        ),
    )


def ensure_unique_vector(
    proposed_vector: List[float],
    registry: set,
    existing_vectors: List[List[float]],
    rng: random.Random,
) -> List[float]:
    if vector_signature(proposed_vector) not in registry:
        return proposed_vector

    for _ in range(50):
        replacement = maximin_immigrant(
            existing_vectors=existing_vectors,
            rng=rng,
            pool_size=100,
        )
        if vector_signature(replacement) not in registry:
            return replacement

    raise RuntimeError("Could not generate a unique AF-MCSO candidate.")


def mcso_rank_key(row: Dict[str, Any]) -> Tuple[float, float, float, str]:
    """Primary objective is validation mAP50-95; other metrics only break ties."""
    return (
        -float(row["val_map50_95"]),
        -float(row["val_map50"]),
        -float(row["val_recall"]),
        str(row["candidate_id"]),
    )


def rank_mcso_rows(rows: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    successful = [
        row for row in rows
        if row.get("status") == "PASS"
        and row.get("val_map50_95", "") != ""
    ]
    return sorted(successful, key=mcso_rank_key)


def build_paired_fidelity_history(
    all_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Build successful 20->40 validation pairs for the same configuration."""
    grouped: Dict[str, Dict[int, Dict[str, Any]]] = defaultdict(dict)

    for row in all_rows:
        if row.get("status") != "PASS" or row.get("val_map50_95", "") == "":
            continue
        budget = int(row["budget_epochs"])
        if budget in {LOW_FIDELITY_EPOCHS, FULL_FIDELITY_EPOCHS}:
            grouped[str(row["candidate_id"])][budget] = row

    history = []
    for candidate_id, by_budget in grouped.items():
        if LOW_FIDELITY_EPOCHS not in by_budget or FULL_FIDELITY_EPOCHS not in by_budget:
            continue

        low_row = by_budget[LOW_FIDELITY_EPOCHS]
        full_row = by_budget[FULL_FIDELITY_EPOCHS]
        map20 = float(low_row["val_map50_95"])
        map40 = float(full_row["val_map50_95"])

        history.append({
            "candidate_id": candidate_id,
            "vector": row_to_vector(low_row),
            "map50_95_20": map20,
            "map50_95_40": map40,
            "gain_20_to_40": map40 - map20,
        })

    return sorted(history, key=lambda item: item["candidate_id"])


def predict_full_fidelity_map50_95(
    low_row: Dict[str, Any],
    paired_history: List[Dict[str, Any]],
) -> Dict[str, float]:
    """
    Predict 40-epoch mAP50-95 using inverse-distance weighted gains from
    completed nearby 20->40 pairs. This affects promotion only.
    """
    observed20 = float(low_row["val_map50_95"])

    if not paired_history:
        return {
            "predicted_gain": 0.0,
            "predicted_map50_95_40": observed20,
            "history_pairs_used": 0,
        }

    target_vector = row_to_vector(low_row)
    distance_rows = [
        (normalized_distance(target_vector, item["vector"]), item)
        for item in paired_history
    ]
    distance_rows.sort(key=lambda pair: pair[0])

    k = min(AF_K_NEIGHBORS, len(distance_rows))
    nearest = distance_rows[:k]

    if nearest[0][0] <= AF_DISTANCE_EPS:
        predicted_gain = float(nearest[0][1]["gain_20_to_40"])
    else:
        weighted_gain = 0.0
        weight_sum = 0.0
        for distance, item in nearest:
            weight = 1.0 / max(distance, AF_DISTANCE_EPS)
            weighted_gain += weight * float(item["gain_20_to_40"])
            weight_sum += weight
        predicted_gain = weighted_gain / weight_sum if weight_sum > 0 else 0.0

    predicted40 = clip01(observed20 + predicted_gain)
    return {
        "predicted_gain": float(predicted_gain),
        "predicted_map50_95_40": float(predicted40),
        "history_pairs_used": int(k),
    }


def score_low_fidelity_candidates(
    low_rows: List[Dict[str, Any]],
    paired_history: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Attach predicted 40-epoch mAP50-95 and rank candidates for promotion."""
    successful = rank_mcso_rows(low_rows)

    if len(paired_history) < AF_MIN_HISTORY_PAIRS:
        raise RuntimeError(
            f"AF promotion requires at least {AF_MIN_HISTORY_PAIRS} completed "
            f"20->40 pairs; found {len(paired_history)}."
        )

    for row in successful:
        prediction = predict_full_fidelity_map50_95(
            low_row=row,
            paired_history=paired_history,
        )
        row["af_predicted_gain"] = prediction["predicted_gain"]
        row["af_predicted_map50_95_40"] = prediction["predicted_map50_95_40"]
        row["af_history_pairs_used"] = prediction["history_pairs_used"]

    return sorted(
        successful,
        key=lambda row: (
            -float(row["af_predicted_map50_95_40"]),
            -float(row["val_map50_95"]),
            -float(row["val_map50"]),
            -float(row["val_recall"]),
            str(row["candidate_id"]),
        ),
    )


def build_full_fidelity_archive(
    full_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    """Keep strong but reasonably distinct 40-epoch configurations as search parents."""
    ranked = rank_mcso_rows(full_rows)
    if not ranked:
        raise RuntimeError("Cannot build an AF-MCSO archive without 40-epoch results.")

    archive: List[Dict[str, Any]] = []
    for row in ranked:
        vector = row_to_vector(row)
        if not archive:
            archive.append(row)
        else:
            min_distance = min(
                normalized_distance(vector, row_to_vector(existing))
                for existing in archive
            )
            if min_distance >= ARCHIVE_MIN_DISTANCE:
                archive.append(row)

        if len(archive) >= ELITE_ARCHIVE_SIZE:
            break

    if len(archive) < min(ELITE_ARCHIVE_SIZE, len(ranked)):
        existing_ids = {row["eval_id"] for row in archive}
        for row in ranked:
            if row["eval_id"] in existing_ids:
                continue
            archive.append(row)
            existing_ids.add(row["eval_id"])
            if len(archive) >= min(ELITE_ARCHIVE_SIZE, len(ranked)):
                break

    return archive


def choose_initial_promotions(
    low_rows: List[Dict[str, Any]],
) -> List[Tuple[Dict[str, Any], str]]:
    """
    Bootstrap AF with two full-fidelity pairs:
    1) best 20-epoch mAP50-95;
    2) a diversity rescue farthest from the first among the remaining candidates.
    """
    ranked = rank_mcso_rows(low_rows)
    if len(ranked) < INITIAL_CONFIRMATIONS:
        raise RuntimeError("Not enough successful Stage-0 candidates for promotion.")

    first = ranked[0]
    remaining = ranked[1:]
    first_vector = row_to_vector(first)
    rescue = max(
        remaining,
        key=lambda row: normalized_distance(row_to_vector(row), first_vector),
    )

    return [
        (first, "initial_best_map50_95"),
        (rescue, "initial_diversity_rescue"),
    ]


def choose_generation_promotion(
    low_rows: List[Dict[str, Any]],
    paired_history: List[Dict[str, Any]],
) -> Tuple[Dict[str, Any], str]:
    ranked = score_low_fidelity_candidates(
        low_rows=low_rows,
        paired_history=paired_history,
    )
    if not ranked:
        raise RuntimeError("No successful generation candidate is available for promotion.")
    return ranked[0], "adaptive_fidelity_predicted_40ep_map50_95"


def generate_generation_proposals(
    *,
    archive_rows: List[Dict[str, Any]],
    evaluated_vectors: List[List[float]],
    registry: set,
    rng: random.Random,
    generation: int,
    previous_generation_improved: bool,
) -> Tuple[List[Tuple[str, List[float]]], str, float]:
    """
    Generate two unique low-fidelity proposals per generation:
    - one elite Levy trial at progressively finer scales;
    - early-stage opposition / maximin exploration, stagnation escape,
      or late-stage recombination of distinct full-fidelity elites.
    """
    if not archive_rows:
        raise RuntimeError("AF-MCSO needs a full-fidelity archive before evolution.")

    base_scale = GENERATION_STEP_SCALES[generation - 1]
    stagnated = generation > 1 and not previous_generation_improved

    if stagnated:
        step_scale = base_scale * STAGNATION_SCALE_MULTIPLIER
        opposition_weight = STAGNATION_OPPOSITION_WEIGHT
        mode = "stagnation_escape"
    else:
        step_scale = base_scale
        opposition_weight = OPPOSITION_DIRECTION_WEIGHT
        mode = "early_exploration" if generation <= 2 else "late_elite_refinement"

    best_parent = row_to_vector(archive_rows[0])
    diverse_parent = row_to_vector(archive_rows[1]) if len(archive_rows) > 1 else best_parent

    exploit = local_levy_child(
        parent=best_parent,
        rng=rng,
        step_scale=step_scale,
    )
    exploit = ensure_unique_vector(
        proposed_vector=exploit,
        registry=registry,
        existing_vectors=evaluated_vectors,
        rng=rng,
    )

    temp_existing = list(evaluated_vectors) + [exploit]

    if stagnated:
        # Escape a measured 40-epoch plateau even in the late local phase.
        explore_role = "stagnation_maximin_immigrant"
        explore = maximin_immigrant(temp_existing, rng, IMMIGRANT_POOL_SIZE)
    elif generation <= 2:
        opposition = opposition_levy_child(
            parent=diverse_parent,
            rng=rng,
            step_scale=step_scale,
            direction_weight=opposition_weight,
        )
        immigrant = maximin_immigrant(temp_existing, rng, IMMIGRANT_POOL_SIZE)

        def min_distance_to_existing(vector: List[float]) -> float:
            return min(normalized_distance(vector, existing) for existing in temp_existing)

        if min_distance_to_existing(opposition) >= min_distance_to_existing(immigrant):
            explore_role, explore = "opposition_levy", opposition
        else:
            explore_role, explore = "maximin_immigrant", immigrant
    else:
        # With more confirmed 20->40 pairs, search locally around observed elites.
        explore_role = "elite_recombination_levy"
        explore = elite_recombination_child(best_parent, diverse_parent, rng, step_scale)

    explore = ensure_unique_vector(
        proposed_vector=explore,
        registry=registry | {vector_signature(exploit)},
        existing_vectors=temp_existing,
        rng=rng,
    )

    return [
        ("elite_levy", exploit),
        (explore_role, explore),
    ], mode, float(step_scale)


def save_rows_to_csv_safe(rows: List[Dict[str, Any]], output_csv: Path) -> None:
    if not rows:
        return
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    fieldnames: List[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)

    with output_csv.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_json(payload: Dict[str, Any], output_path: Path) -> None:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def summarize_numeric_rows(
    rows: List[Dict[str, Any]],
    metric_names: List[str],
) -> List[Dict[str, Any]]:
    summary_rows = []
    for metric_name in metric_names:
        values = [float(row[metric_name]) for row in rows if row.get(metric_name, "") != ""]
        if not values:
            continue
        summary_rows.append({
            "metric": metric_name,
            "n": len(values),
            "mean": float(mean(values)),
            "sd": float(stdev(values)) if len(values) >= 2 else "",
            "min": float(min(values)),
            "max": float(max(values)),
        })
    return summary_rows


def summarize_per_class_rows(
    per_class_rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    grouped: Dict[Tuple[int, str], List[float]] = defaultdict(list)
    for row in per_class_rows:
        grouped[(int(row["class_id"]), str(row["class_name"]))].append(
            float(row["test_map50_95"])
        )

    output = []
    for (class_id, class_name), values in sorted(grouped.items()):
        output.append({
            "class_id": class_id,
            "class_name": class_name,
            "n": len(values),
            "mean_test_map50_95": float(mean(values)),
            "sd_test_map50_95": float(stdev(values)) if len(values) >= 2 else "",
            "min_test_map50_95": float(min(values)),
            "max_test_map50_95": float(max(values)),
        })
    return output


def build_boundary_report(config: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []
    vector = encode_config(config)
    for parameter_name, z in zip(SEARCH_PARAMETER_ORDER, vector):
        spec = SEARCH_SPACE[parameter_name]
        rows.append({
            "parameter": parameter_name,
            "selected_value": float(config[parameter_name]),
            "low": float(spec["low"]),
            "high": float(spec["high"]),
            "distribution": spec["distribution"],
            "normalized_position": float(z),
            "distance_to_nearest_boundary": float(min(z, 1.0 - z)),
        })
    return rows


## Module 8: AF-MCSO exact 600-unit optimizer (unmodified search mechanics)

Each fresh direction runs 4×20 + 2×40 + 5×(2×20 + 1×40) + 1×40 = **600 source-validation HPO epoch-units**. 14 low-fidelity and 8 full-fidelity evaluations, including the final global rescue; only measured full-fidelity scores can win.


In [ ]:
def reserve_hpo_budget(
    budget_tracker: Dict[str, int],
    budget_epochs: int,
) -> None:
    """Count a scheduled evaluation before training so failures cannot be silently replaced."""
    budget_epochs = int(budget_epochs)
    projected = budget_tracker["attempted_epoch_units"] + budget_epochs
    if projected > PLANNED_HPO_EPOCH_UNITS:
        raise RuntimeError(
            f"AF-MCSO would exceed the frozen {PLANNED_HPO_EPOCH_UNITS}-unit HPO budget."
        )

    budget_tracker["attempted_epoch_units"] = projected
    budget_tracker["attempted_evaluations"] += 1
    if budget_epochs == LOW_FIDELITY_EPOCHS:
        budget_tracker["attempted_20epoch"] += 1
    elif budget_epochs == FULL_FIDELITY_EPOCHS:
        budget_tracker["attempted_40epoch"] += 1
    else:
        raise ValueError(f"Unexpected AF-MCSO fidelity: {budget_epochs}")


def evaluate_mcso_candidate(
    *,
    candidate_id: str,
    vector: List[float],
    budget_epochs: int,
    stage_name: str,
    generation: int,
    role: str,
    promotion_reason: str,
    optimization_dataset_yaml: Path,
    device: str,
    budget_tracker: Dict[str, int],
) -> Dict[str, Any]:
    """Evaluate one AF-MCSO configuration at 20 or 40 epochs."""
    reserve_hpo_budget(budget_tracker, budget_epochs)
    evaluation_start = time.perf_counter()

    config = decode_vector(vector, config_id=candidate_id)
    warmup_epochs, close_mosaic = get_fidelity_controls(budget_epochs)

    try:
        result = train_and_validate_configuration(
            learning_rate=config["learning_rate"],
            lrf=config["lrf"],
            momentum=config["momentum"],
            weight_decay=config["weight_decay"],
            scale=config["scale"],
            dataset_yaml=optimization_dataset_yaml,
            project_dir=HPO_RUNS_ROOT,
            model_name=MODEL_NAME,
            epochs=int(budget_epochs),
            image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE,
            patience=PATIENCE,
            workers=WORKERS,
            seed=OPTIMIZATION_TRAIN_SEED,
            warmup_epochs=float(warmup_epochs),
            close_mosaic=int(close_mosaic),
            fixed_training_args=FIXED_TRAINING_ARGS,
            expected_class_names=CLASS_NAMES,
            strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
            fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE,
            device=device,
        )

        metrics = result["metrics"]
        wall_seconds = time.perf_counter() - evaluation_start

        row = {
            "status": "PASS",
            "dataset_tag": DATASET_TAG,
            "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR, "test_year": TEST_YEAR,
            "eval_id": f"{candidate_id}_E{int(budget_epochs)}",
            "candidate_id": candidate_id,
            "stage": stage_name,
            "generation": int(generation),
            "role": role,
            "promotion_reason": promotion_reason,
            "budget_epochs": int(budget_epochs),
            "learning_rate": float(config["learning_rate"]),
            "lrf": float(config["lrf"]),
            "momentum": float(config["momentum"]),
            "weight_decay": float(config["weight_decay"]),
            "scale": float(config["scale"]),
            "val_precision": float(metrics["precision"]),
            "val_recall": float(metrics["recall"]),
            "val_map50": float(metrics["map50"]),
            "val_map50_95": float(metrics["map50_95"]),
            "val_bottom3_mean_map50_95": float(metrics["bottom3_mean_map50_95"]),
            "training_seconds": float(result["timing"]["training_seconds"]),
            "validation_seconds": float(result["timing"]["validation_seconds"]),
            "evaluation_wall_seconds": float(wall_seconds),
            "af_predicted_gain": "",
            "af_predicted_map50_95_40": "",
            "af_history_pairs_used": "",
            "af_observed_gain": "",
            "best_checkpoint": result["paths"]["best_checkpoint"],
            "error_message": "",
        }
    except Exception as error:
        wall_seconds = time.perf_counter() - evaluation_start
        print("Evaluation failed:", sanitize_exception_message(error))

        row = {
            "status": "FAIL",
            "dataset_tag": DATASET_TAG,
            "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR, "test_year": TEST_YEAR,
            "eval_id": f"{candidate_id}_E{int(budget_epochs)}",
            "candidate_id": candidate_id,
            "stage": stage_name,
            "generation": int(generation),
            "role": role,
            "promotion_reason": promotion_reason,
            "budget_epochs": int(budget_epochs),
            "learning_rate": float(config["learning_rate"]),
            "lrf": float(config["lrf"]),
            "momentum": float(config["momentum"]),
            "weight_decay": float(config["weight_decay"]),
            "scale": float(config["scale"]),
            "val_precision": "",
            "val_recall": "",
            "val_map50": "",
            "val_map50_95": "",
            "val_bottom3_mean_map50_95": "",
            "training_seconds": "",
            "validation_seconds": "",
            "evaluation_wall_seconds": float(wall_seconds),
            "af_predicted_gain": "",
            "af_predicted_map50_95_40": "",
            "af_history_pairs_used": "",
            "af_observed_gain": "",
            "best_checkpoint": "",
            "error_message": sanitize_exception_message(error),
        }

    for index, value in enumerate(vector):
        row[f"z{index}"] = float(value)

    return row


def confirm_promoted_candidates(
    *,
    selected_pairs: List[Tuple[Dict[str, Any], str]],
    stage_name: str,
    generation: int,
    optimization_dataset_yaml: Path,
    device: str,
    all_rows: List[Dict[str, Any]],
    promotion_rows: List[Dict[str, Any]],
    results_csv: Path,
    promotions_csv: Path,
    budget_tracker: Dict[str, int],
) -> List[Dict[str, Any]]:
    """Retrain selected 20-epoch configurations independently at 40 epochs."""
    confirmed_rows = []

    for low_row, reason in selected_pairs:
        full_row = evaluate_mcso_candidate(
            candidate_id=str(low_row["candidate_id"]),
            vector=row_to_vector(low_row),
            budget_epochs=FULL_FIDELITY_EPOCHS,
            stage_name=stage_name,
            generation=generation,
            role=str(low_row["role"]),
            promotion_reason=reason,
            optimization_dataset_yaml=optimization_dataset_yaml,
            device=device,
            budget_tracker=budget_tracker,
        )

        full_row["af_predicted_gain"] = low_row.get("af_predicted_gain", "")
        full_row["af_predicted_map50_95_40"] = low_row.get(
            "af_predicted_map50_95_40", ""
        )
        full_row["af_history_pairs_used"] = low_row.get("af_history_pairs_used", "")

        if full_row["status"] == "PASS":
            observed_gain = (
                float(full_row["val_map50_95"]) - float(low_row["val_map50_95"])
            )
            low_row["af_observed_gain"] = observed_gain
            full_row["af_observed_gain"] = observed_gain

        all_rows.append(full_row)
        confirmed_rows.append(full_row)

        promotion_rows.append({
            "candidate_id": low_row["candidate_id"],
            "stage": stage_name,
            "generation": int(generation),
            "promotion_reason": reason,
            "map50_95_20": low_row.get("val_map50_95", ""),
            "predicted_gain_20_to_40": low_row.get("af_predicted_gain", ""),
            "predicted_map50_95_40": low_row.get("af_predicted_map50_95_40", ""),
            "history_pairs_used": low_row.get("af_history_pairs_used", ""),
            "map50_95_40": full_row.get("val_map50_95", ""),
            "observed_gain_20_to_40": full_row.get("af_observed_gain", ""),
            "status_40": full_row["status"],
        })

        save_rows_to_csv_safe(all_rows, results_csv)
        save_rows_to_csv_safe(promotion_rows, promotions_csv)

    return confirmed_rows


def require_all_pass(rows: List[Dict[str, Any]], context: str) -> None:
    failed = [row for row in rows if row.get("status") != "PASS"]
    if failed:
        failed_ids = [row["eval_id"] for row in failed]
        raise RuntimeError(
            f"{context} has failed scheduled evaluations: {failed_ids}. "
            "They are not silently replaced because the HPO budget is frozen."
        )


def run_af_mcso_hpo(
    *,
    optimization_dataset_yaml: Path,
    device: str,
    results_csv: Path,
    promotions_csv: Path,
    stage_summary_csv: Path,
) -> Tuple[
    Dict[str, Any],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
    List[Dict[str, Any]],
]:
    """Run the exact per-year 600-unit AF-MCSO HPO schedule."""
    rng = random.Random(MCSO_RANDOM_SEED)

    all_rows: List[Dict[str, Any]] = []
    promotion_rows: List[Dict[str, Any]] = []
    stage_summary_rows: List[Dict[str, Any]] = []

    vector_registry = set()
    evaluated_vectors: List[List[float]] = []
    candidate_counter = 0

    budget_tracker = {
        "attempted_epoch_units": 0,
        "attempted_evaluations": 0,
        "attempted_20epoch": 0,
        "attempted_40epoch": 0,
    }

    # ========================================================
    # Stage 0: diverse 20-epoch initialization + two confirmations
    # ========================================================
    print("\n" + "=" * 100)
    print("AF-MCSO STAGE 0 : 4 MAXIMIN CANDIDATES @20 -> 2 CONFIRMATIONS @40")
    print("=" * 100)

    initial_vectors = maximin_initial_vectors(
        rng=rng,
        n_vectors=INITIAL_NESTS,
        pool_size=INITIAL_MAXIMIN_POOL,
    )

    initial_low_rows = []
    for vector in initial_vectors:
        candidate_counter += 1
        candidate_id = f"N{candidate_counter:03d}"

        signature = vector_signature(vector)
        if signature in vector_registry:
            raise RuntimeError("Duplicate Stage-0 vector.")
        vector_registry.add(signature)
        evaluated_vectors.append(vector)

        row = evaluate_mcso_candidate(
            candidate_id=candidate_id,
            vector=vector,
            budget_epochs=LOW_FIDELITY_EPOCHS,
            stage_name="stage0_initial",
            generation=0,
            role="maximin_initial",
            promotion_reason="",
            optimization_dataset_yaml=optimization_dataset_yaml,
            device=device,
            budget_tracker=budget_tracker,
        )
        all_rows.append(row)
        initial_low_rows.append(row)
        save_rows_to_csv_safe(all_rows, results_csv)

    require_all_pass(initial_low_rows, "Stage-0 low-fidelity initialization")

    initial_promotions = choose_initial_promotions(initial_low_rows)
    initial_full_rows = confirm_promoted_candidates(
        selected_pairs=initial_promotions,
        stage_name="stage0_initial",
        generation=0,
        optimization_dataset_yaml=optimization_dataset_yaml,
        device=device,
        all_rows=all_rows,
        promotion_rows=promotion_rows,
        results_csv=results_csv,
        promotions_csv=promotions_csv,
        budget_tracker=budget_tracker,
    )
    require_all_pass(initial_full_rows, "Stage-0 full-fidelity confirmations")

    paired_history = build_paired_fidelity_history(all_rows)
    if len(paired_history) != INITIAL_CONFIRMATIONS:
        raise RuntimeError(
            f"Stage 0 should create {INITIAL_CONFIRMATIONS} completed 20->40 pairs; "
            f"found {len(paired_history)}."
        )

    initial_diversity = mean_pairwise_distance(
        [row_to_vector(row) for row in initial_low_rows]
    )
    incumbent = float(rank_mcso_rows(initial_full_rows)[0]["val_map50_95"])

    stage_summary_rows.append({
        "stage": "stage0_initial",
        "generation": 0,
        "mode": "initialization",
        "candidate_count_20": len(initial_low_rows),
        "confirmation_count_40": len(initial_full_rows),
        "mean_pairwise_distance_20": initial_diversity,
        "best_map50_95_20": float(rank_mcso_rows(initial_low_rows)[0]["val_map50_95"]),
        "best_confirmed_map50_95_40": incumbent,
        "incumbent_map50_95_40": incumbent,
        "paired_fidelity_history_size": len(paired_history),
        "best_predicted_map50_95_40": "",
        "improved_full_fidelity": True,
    })
    save_rows_to_csv_safe(stage_summary_rows, stage_summary_csv)

    previous_generation_improved = True

    # ========================================================
    # Generations 1-5: two 20-epoch proposals -> one AF promotion
    # ========================================================
    for generation in range(1, MCSO_GENERATIONS + 1):
        print("\n" + "#" * 100)
        print(f"AF-MCSO GENERATION {generation}/{MCSO_GENERATIONS}")
        print("#" * 100)

        successful_full_rows = [
            row for row in all_rows
            if row["status"] == "PASS"
            and int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
        ]
        archive = build_full_fidelity_archive(successful_full_rows)
        incumbent_before = float(rank_mcso_rows(successful_full_rows)[0]["val_map50_95"])

        proposals, mode, step_scale = generate_generation_proposals(
            archive_rows=archive,
            evaluated_vectors=evaluated_vectors,
            registry=vector_registry,
            rng=rng,
            generation=generation,
            previous_generation_improved=previous_generation_improved,
        )

        generation_low_rows = []
        for role, vector in proposals:
            signature = vector_signature(vector)
            if signature in vector_registry:
                raise RuntimeError(f"Duplicate proposal produced in generation {generation}.")
            vector_registry.add(signature)
            evaluated_vectors.append(vector)

            candidate_counter += 1
            candidate_id = f"G{generation}C{candidate_counter:03d}"

            row = evaluate_mcso_candidate(
                candidate_id=candidate_id,
                vector=vector,
                budget_epochs=LOW_FIDELITY_EPOCHS,
                stage_name=f"generation_{generation}",
                generation=generation,
                role=role,
                promotion_reason="",
                optimization_dataset_yaml=optimization_dataset_yaml,
                device=device,
                budget_tracker=budget_tracker,
            )
            all_rows.append(row)
            generation_low_rows.append(row)
            save_rows_to_csv_safe(all_rows, results_csv)

        require_all_pass(generation_low_rows, f"Generation {generation} low-fidelity search")

        # Only earlier completed pairs are used to predict the candidates' 40-epoch performance.
        paired_history = build_paired_fidelity_history(all_rows)
        ranked_for_promotion = score_low_fidelity_candidates(
            low_rows=generation_low_rows,
            paired_history=paired_history,
        )

        print("\nAdaptive-fidelity promotion ranking:")
        for rank, row in enumerate(ranked_for_promotion, start=1):
            print(
                f"#{rank} {row['candidate_id']} | "
                f"mAP95@20={float(row['val_map50_95']):.6f} | "
                f"pred_gain={float(row['af_predicted_gain']):+.6f} | "
                f"pred_mAP95@40={float(row['af_predicted_map50_95_40']):.6f}"
            )

        promoted_row, reason = choose_generation_promotion(
            low_rows=generation_low_rows,
            paired_history=paired_history,
        )
        confirmed_rows = confirm_promoted_candidates(
            selected_pairs=[(promoted_row, reason)],
            stage_name=f"generation_{generation}",
            generation=generation,
            optimization_dataset_yaml=optimization_dataset_yaml,
            device=device,
            all_rows=all_rows,
            promotion_rows=promotion_rows,
            results_csv=results_csv,
            promotions_csv=promotions_csv,
            budget_tracker=budget_tracker,
        )
        require_all_pass(confirmed_rows, f"Generation {generation} full-fidelity confirmation")

        successful_full_rows = [
            row for row in all_rows
            if row["status"] == "PASS"
            and int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
        ]
        incumbent_after = float(rank_mcso_rows(successful_full_rows)[0]["val_map50_95"])
        improvement = incumbent_after - incumbent_before
        previous_generation_improved = improvement > STAGNATION_TOLERANCE

        generation_diversity = mean_pairwise_distance(
            [row_to_vector(row) for row in generation_low_rows]
        )

        stage_summary_rows.append({
            "stage": f"generation_{generation}",
            "generation": generation,
            "mode": mode,
            "step_scale": step_scale,
            "candidate_count_20": len(generation_low_rows),
            "confirmation_count_40": len(confirmed_rows),
            "mean_pairwise_distance_20": generation_diversity,
            "best_map50_95_20": float(rank_mcso_rows(generation_low_rows)[0]["val_map50_95"]),
            "best_confirmed_map50_95_40": float(confirmed_rows[0]["val_map50_95"]),
            "incumbent_map50_95_40": incumbent_after,
            "paired_fidelity_history_size": len(build_paired_fidelity_history(all_rows)),
            "best_predicted_map50_95_40": max(
                float(row["af_predicted_map50_95_40"])
                for row in generation_low_rows
            ),
            "improved_full_fidelity": previous_generation_improved,
        })
        save_rows_to_csv_safe(stage_summary_rows, stage_summary_csv)

        print(f"Incumbent before: {incumbent_before:.6f}")
        print(f"Incumbent after:  {incumbent_after:.6f}")
        print(f"Improvement:      {improvement:+.6f}")

    # ========================================================
    # Final 40-epoch archive rescue: reconsider ALL earlier skipped
    # 20-epoch configurations using the now larger paired history.
    # A small novelty bonus makes the final confirmation less redundant.
    # ========================================================
    print("\n" + "=" * 100)
    print("AF-MCSO FINAL GLOBAL RESCUE: 1 SKIPPED 20-EPOCH CANDIDATE -> 40 EPOCHS")
    print("=" * 100)
    previous_full_rows = [
        row for row in all_rows
        if row["status"] == "PASS"
        and int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
    ]
    before_rescue = float(rank_mcso_rows(previous_full_rows)[0]["val_map50_95"])
    previously_promoted_ids = {row["candidate_id"] for row in previous_full_rows}
    skipped_low_rows = [
        row for row in all_rows
        if row["status"] == "PASS"
        and int(row["budget_epochs"]) == LOW_FIDELITY_EPOCHS
        and row["candidate_id"] not in previously_promoted_ids
    ]
    if len(skipped_low_rows) < FINAL_RESCUE_PROMOTIONS:
        raise RuntimeError("Not enough unpromoted 20-epoch candidates for final rescue.")
    paired_history = build_paired_fidelity_history(all_rows)
    rescue_candidates = score_low_fidelity_candidates(skipped_low_rows, paired_history)
    for row in rescue_candidates:
        nearest_confirmed_distance = min(
            normalized_distance(row_to_vector(row), row_to_vector(full_row))
            for full_row in previous_full_rows
        )
        row["af_rescue_novelty"] = float(nearest_confirmed_distance)
        row["af_rescue_score"] = (
            float(row["af_predicted_map50_95_40"])
            + FINAL_RESCUE_NOVELTY_WEIGHT * nearest_confirmed_distance
        )
    rescue_candidates.sort(key=lambda row: (
        -float(row["af_rescue_score"]),
        -float(row["af_predicted_map50_95_40"]),
        str(row["candidate_id"]),
    ))
    rescue_selected = rescue_candidates[:FINAL_RESCUE_PROMOTIONS]
    print("Final rescue shortlist:")
    for row in rescue_selected:
        print(
            f"{row['candidate_id']} | observed@20={float(row['val_map50_95']):.6f} "
            f"| predicted@40={float(row['af_predicted_map50_95_40']):.6f} "
            f"| novelty={float(row['af_rescue_novelty']):.4f}"
        )
    rescue_confirmed_rows = confirm_promoted_candidates(
        selected_pairs=[
            (row, "final_global_rescue_predicted40_plus_novelty")
            for row in rescue_selected
        ],
        stage_name="final_global_rescue",
        generation=MCSO_GENERATIONS + 1,
        optimization_dataset_yaml=optimization_dataset_yaml,
        device=device,
        all_rows=all_rows,
        promotion_rows=promotion_rows,
        results_csv=results_csv,
        promotions_csv=promotions_csv,
        budget_tracker=budget_tracker,
    )
    require_all_pass(rescue_confirmed_rows, "Final global rescue confirmation")
    after_rescue = max(
        float(row["val_map50_95"])
        for row in previous_full_rows + rescue_confirmed_rows
    )
    stage_summary_rows.append({
        "stage": "final_global_rescue",
        "generation": MCSO_GENERATIONS + 1,
        "mode": "skipped_candidate_reconsideration",
        "step_scale": "",
        "candidate_count_20": 0,
        "confirmation_count_40": len(rescue_confirmed_rows),
        "skipped_candidates_scored": len(skipped_low_rows),
        "mean_pairwise_distance_20": "",
        "best_map50_95_20": float(rank_mcso_rows(skipped_low_rows)[0]["val_map50_95"]),
        "best_confirmed_map50_95_40": float(rescue_confirmed_rows[0]["val_map50_95"]),
        "incumbent_map50_95_40": after_rescue,
        "paired_fidelity_history_size": len(build_paired_fidelity_history(all_rows)),
        "best_predicted_map50_95_40": max(
            float(row["af_predicted_map50_95_40"]) for row in rescue_selected
        ),
        "improved_full_fidelity": after_rescue > before_rescue + STAGNATION_TOLERANCE,
    })
    save_rows_to_csv_safe(stage_summary_rows, stage_summary_csv)

    # ========================================================
    # Exact-budget audit and full-fidelity-only winner
    # ========================================================
    successful_rows = [row for row in all_rows if row["status"] == "PASS"]
    actual_successful_units = sum(int(row["budget_epochs"]) for row in successful_rows)

    if REQUIRE_FULL_HPO_BUDGET:
        if budget_tracker["attempted_epoch_units"] != PLANNED_HPO_EPOCH_UNITS:
            raise RuntimeError(
                f"Expected {PLANNED_HPO_EPOCH_UNITS} attempted HPO epoch-units, "
                f"got {budget_tracker['attempted_epoch_units']}."
            )
        if budget_tracker["attempted_20epoch"] != EXPECTED_LOW_FIDELITY_EVALUATIONS:
            raise RuntimeError("20-epoch evaluation count does not match the frozen plan.")
        if budget_tracker["attempted_40epoch"] != EXPECTED_FULL_FIDELITY_EVALUATIONS:
            raise RuntimeError("40-epoch evaluation count does not match the frozen plan.")
        if len(successful_rows) != EXPECTED_TOTAL_HPO_EVALUATIONS:
            raise RuntimeError(
                f"Expected {EXPECTED_TOTAL_HPO_EVALUATIONS} successful HPO evaluations, "
                f"got {len(successful_rows)}."
            )
        if actual_successful_units != PLANNED_HPO_EPOCH_UNITS:
            raise RuntimeError(
                f"Expected {PLANNED_HPO_EPOCH_UNITS} successful HPO epoch-units, "
                f"got {actual_successful_units}."
            )

    full_rows = [
        row for row in successful_rows
        if int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS
    ]
    if len(full_rows) != EXPECTED_FULL_FIDELITY_EVALUATIONS:
        raise RuntimeError(
            f"Expected {EXPECTED_FULL_FIDELITY_EVALUATIONS} successful 40-epoch evaluations, "
            f"got {len(full_rows)}."
        )

    ranked_full = rank_mcso_rows(full_rows)
    best_row = ranked_full[0]

    selected_config = {
        "method": "Adaptive Fidelity-Aware Modified Cuckoo Search (AF-MCSO-600)",
        "dataset_tag": DATASET_TAG,
        "selection_rule": (
            "best successful 40-epoch validation mAP50-95; "
            "ties by mAP50, recall, candidate_id"
        ),
        "test_set_used_for_selection": False,
        "selected_candidate_id": best_row["candidate_id"],
        "selected_stage": best_row["stage"],
        "selected_generation": int(best_row["generation"]),
        "selected_role": best_row["role"],
        "selected_validation_map50_95": float(best_row["val_map50_95"]),
        "selected_validation_map50": float(best_row["val_map50"]),
        "selected_validation_recall": float(best_row["val_recall"]),
        "learning_rate": float(best_row["learning_rate"]),
        "lrf": float(best_row["lrf"]),
        "momentum": float(best_row["momentum"]),
        "weight_decay": float(best_row["weight_decay"]),
        "scale": float(best_row["scale"]),
        "mcso_random_seed": int(MCSO_RANDOM_SEED),
        "levy_beta": float(LEVY_BETA),
        "generations": int(MCSO_GENERATIONS),
        "final_global_rescue_promotions": int(FINAL_RESCUE_PROMOTIONS),
        "final_rescue_novelty_weight": float(FINAL_RESCUE_NOVELTY_WEIGHT),
        "adaptive_fidelity_k_neighbors": int(AF_K_NEIGHBORS),
        "adaptive_fidelity_role": (
            "promotion only; final HPO selection uses actual 40-epoch validation mAP50-95"
        ),
        "planned_hpo_epoch_units": int(PLANNED_HPO_EPOCH_UNITS),
        "attempted_hpo_epoch_units": int(budget_tracker["attempted_epoch_units"]),
        "actual_successful_hpo_epoch_units": int(actual_successful_units),
        "hpo_successful_evaluations": int(len(successful_rows)),
        "hpo_full_fidelity_evaluations": int(len(full_rows)),
        "final_epochs": int(FINAL_EPOCHS),
        "search_space": SEARCH_SPACE,
    }

    print("\n" + "=" * 100)
    print("AF-MCSO HPO COMPLETE")
    print("=" * 100)
    print("Successful HPO evaluations:", len(successful_rows))
    print("20-epoch evaluations:", EXPECTED_LOW_FIDELITY_EVALUATIONS)
    print("40-epoch evaluations:", len(full_rows))
    print("HPO epoch-units:", actual_successful_units)
    print("Selected:", selected_config["selected_candidate_id"])
    print("Selected validation mAP50-95:", selected_config["selected_validation_map50_95"])
    print("Test set used for selection: NO")
    print("=" * 100)

    return selected_config, all_rows, promotion_rows, stage_summary_rows


## Module 9: Source-validation AF-MCSO diagnostic plot

The optimizer progress chart reports only the source validation set, never the target test set.


In [ ]:
def plot_af_mcso_diagnostics(
    *,
    search_rows: List[Dict[str, Any]],
    promotion_rows: List[Dict[str, Any]],
    stage_summary_rows: List[Dict[str, Any]],
) -> None:
    """Save one compact plot of AF-MCSO validation progress."""
    del promotion_rows, stage_summary_rows

    successful_rows = [row for row in search_rows if row.get("status") == "PASS"]
    if not successful_rows:
        return

    RUN_PLOTS_ROOT.mkdir(parents=True, exist_ok=True)

    evaluation_index = list(range(1, len(successful_rows) + 1))
    values = [float(row["val_map50_95"]) for row in successful_rows]
    fidelities = [int(row["budget_epochs"]) for row in successful_rows]

    incumbent_values: List[float] = []
    incumbent = -float("inf")
    for row in successful_rows:
        if int(row["budget_epochs"]) == FULL_FIDELITY_EPOCHS:
            incumbent = max(incumbent, float(row["val_map50_95"]))
        incumbent_values.append(np.nan if not math.isfinite(incumbent) else incumbent)

    plt.figure(figsize=(9, 5))
    low_x = [x for x, fidelity in zip(evaluation_index, fidelities) if fidelity == LOW_FIDELITY_EPOCHS]
    low_y = [y for y, fidelity in zip(values, fidelities) if fidelity == LOW_FIDELITY_EPOCHS]
    full_x = [x for x, fidelity in zip(evaluation_index, fidelities) if fidelity == FULL_FIDELITY_EPOCHS]
    full_y = [y for y, fidelity in zip(values, fidelities) if fidelity == FULL_FIDELITY_EPOCHS]

    if low_x:
        plt.scatter(low_x, low_y, marker="o", label="20-epoch evaluation")
    if full_x:
        plt.scatter(full_x, full_y, marker="s", label="40-epoch evaluation")
    if any(math.isfinite(float(v)) for v in incumbent_values if not np.isnan(v)):
        plt.plot(evaluation_index, incumbent_values, linewidth=1.5, label="40-epoch incumbent")

    plt.xlabel("HPO evaluation order")
    plt.ylabel("Validation mAP50-95")
    plt.title(f"{DATASET_TAG}: AF-MCSO validation progress")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        RUN_PLOTS_ROOT / "af_mcso_validation_progress.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()


## Module 10: Frozen checkpoint evaluation on opposite-year test only

Reuses the original AF-MCSO test-only evaluator. It is invoked **after** final training or for validated checkpoints in inference-only reuse mode.


In [ ]:
def evaluate_checkpoint_on_test_split(
    *,
    checkpoint_path: Path,
    dataset_yaml: Path,
    output_root: Path,
    run_name: str,
    expected_class_names: List[str],
    image_size: int,
    batch_size: int,
    workers: int,
    device: Optional[str] = None,
) -> Dict[str, Any]:
    """Evaluate one frozen checkpoint on the test split only."""
    checkpoint_path = Path(checkpoint_path)
    dataset_yaml = Path(dataset_yaml)
    output_root = Path(output_root)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset YAML not found: {dataset_yaml}")

    if device is None:
        device = "0" if torch.cuda.is_available() else "cpu"

    output_root.mkdir(parents=True, exist_ok=True)
    model = YOLO(str(checkpoint_path))

    test_start = time.perf_counter()
    test_metrics = model.val(
        data=str(dataset_yaml.resolve()),
        split="test",
        imgsz=int(image_size),
        batch=int(batch_size),
        workers=int(workers),
        device=device,
        project=str(output_root.resolve()),
        name=run_name,
        plots=True,
        save_json=False,
        verbose=False,
    )
    test_seconds = time.perf_counter() - test_start

    metric_data = extract_box_metrics(
        metrics=test_metrics,
        expected_class_names=expected_class_names,
    )

    per_class_rows = [
        {
            "class_id": row["class_id"],
            "class_name": row["class_name"],
            "test_map50_95": row["map50_95"],
        }
        for row in metric_data["per_class_map50_95"]
    ]

    return {
        "test_precision": metric_data["precision"],
        "test_recall": metric_data["recall"],
        "test_map50": metric_data["map50"],
        "test_map50_95": metric_data["map50_95"],
        "test_bottom3_mean_map50_95": metric_data["bottom3_mean_map50_95"],
        "test_seconds": float(test_seconds),
        "per_class_rows": per_class_rows,
    }

## Module 11: Fresh AF-MCSO cross-year workflow (600 HPO + three final seeds)

For either direction, build the source train/validation and target test split at optimization seed 42, run the unmodified AF-MCSO search on source validation only, freeze the best observed 40-epoch candidate, retrain from `yolov8m.pt` for seeds 42/123/999, and test each frozen checkpoint only on target-year test. HPO and final seed training are separately budgeted.


In [ ]:
def scan_cross_year_records() -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], Dict[str, Any]]:
    """Read/audit two independent years. No target annotations enter training selection."""
    source_records, source_stats = build_clean_records(TRAIN_SOURCE_ROOT, TARGET_CLASS_SET, LABEL_ALIASES)
    target_records, target_stats = build_clean_records(TEST_TARGET_ROOT, TARGET_CLASS_SET, LABEL_ALIASES)
    source_audit, target_audit = audit_records(source_records, CLASS_NAMES), audit_records(target_records, CLASS_NAMES)
    for role, audit in (("source", source_audit), ("target", target_audit)):
        if audit["missing_classes"]:
            raise RuntimeError(f"Missing classes in {role} year: {audit['missing_classes']}")
    save_json({"direction": CROSS_YEAR_TAG, "source_stats": source_stats,
               "target_stats": target_stats, "source_audit": source_audit,
               "target_audit": target_audit}, RUN_STATS_ROOT / "cross_year_dataset_audit.json")
    return source_records, target_records, {
        "source_stats": source_stats, "target_stats": target_stats,
        "source_audit": source_audit, "target_audit": target_audit,
    }


def validate_fresh_af_mcso_budget(rows: List[Dict[str, Any]]) -> Dict[str, int]:
    passed = [r for r in rows if r.get("status") == "PASS"]
    low = sum(int(r["budget_epochs"]) == LOW_FIDELITY_EPOCHS for r in passed)
    full = sum(int(r["budget_epochs"]) == FULL_FIDELITY_EPOCHS for r in passed)
    units = sum(int(r["budget_epochs"]) for r in passed)
    expected = (EXPECTED_LOW_FIDELITY_EVALUATIONS, EXPECTED_FULL_FIDELITY_EVALUATIONS,
                EXPECTED_TOTAL_HPO_EVALUATIONS, PLANNED_HPO_EPOCH_UNITS)
    if (low, full, len(passed), units) != expected or len(rows) != len(passed):
        raise RuntimeError(f"Incomplete AF-MCSO 600-unit search: got {(low, full, len(passed), units)}; need {expected}")
    return {"low_fidelity_evaluations": low, "full_fidelity_evaluations": full,
            "total_evaluations": len(passed), "hpo_epoch_units": units}


def build_af_mcso_target_summary(final_rows: List[Dict[str, Any]], *,
                                   new_hpo_units: int, new_final_units: int,
                                   historical_selection_seconds: float = 0.0) -> List[Dict[str, Any]]:
    columns = ["target_test_precision", "target_test_recall", "target_test_map50",
               "target_test_map50_95", "target_test_bottom3_mean_map50_95", "target_test_seconds",
               "final_run_wall_seconds", "final_training_seconds", "final_source_validation_seconds"]
    result = summarize_numeric_rows(final_rows, columns)
    for name, value in (("hpo_epoch_units", PLANNED_HPO_EPOCH_UNITS),
                        ("new_hpo_epoch_units", new_hpo_units),
                        ("new_final_training_epoch_units", new_final_units),
                        ("historical_selection_stage_wall_seconds", historical_selection_seconds)):
        result.append({"metric": name, "n": 1, "mean": value, "sd": "", "min": value, "max": value})
    return result


def save_target_test_plot(final_rows: List[Dict[str, Any]]) -> None:
    metrics = [("Precision", "target_test_precision"), ("Recall", "target_test_recall"),
               ("mAP50", "target_test_map50"), ("mAP50–95", "target_test_map50_95")]
    vals = [[float(row[key]) for row in final_rows] for _, key in metrics]
    plt.figure(figsize=(8, 5))
    plt.bar([label for label, _ in metrics], [mean(a) for a in vals],
            yerr=[stdev(a) if len(a) > 1 else 0.0 for a in vals], capsize=4)
    plt.ylim(0.0, 1.0)
    plt.ylabel("Opposite-year test score")
    plt.title(f"{CROSS_YEAR_TAG}: AF-MCSO 3-seed target-test mean ± SD")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    RUN_PLOTS_ROOT.mkdir(parents=True, exist_ok=True)
    plt.savefig(RUN_PLOTS_ROOT / "af_mcso_cross_year_target_test_mean_sd.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()


def run_fresh_cross_year_af_mcso() -> Dict[str, Any]:
    """Use source-year ONLY to optimize, then test three frozen seeds on target year."""
    ensure_output_directories()
    source_records, target_records, audit = scan_cross_year_records()
    device = "0" if torch.cuda.is_available() else "cpu"
    workflow_start = time.perf_counter()
    print(f"FRESH AF-MCSO cross-year run: {CROSS_YEAR_TAG}")

    optimization_cross, optimization_source, optimization_target, optimization_info = (
        make_cross_year_splits_for_seed(OPTIMIZATION_SPLIT_SEED, source_records, target_records)
    )
    optimization_yaml, _ = prepare_cross_year_yolo_dataset(
        cross_year_splits=optimization_cross, source_splits=optimization_source,
        target_splits=optimization_target, split_info=optimization_info,
        output_root=YOLO_DATA_ROOT / f"optimization_seed{OPTIMIZATION_SPLIT_SEED}",
        manifest_root=SPLITS_ROOT / CROSS_YEAR_TAG / f"optimization_seed{OPTIMIZATION_SPLIT_SEED}",
        image_mode=IMAGE_PLACEMENT_MODE, seed=OPTIMIZATION_SPLIT_SEED,
    )
    selection_start = time.perf_counter()
    selected_config, search_rows, promotion_rows, stage_summary_rows = run_af_mcso_hpo(
        optimization_dataset_yaml=optimization_yaml, device=device,
        results_csv=RUN_STATS_ROOT / "af_mcso_source_validation_results.csv",
        promotions_csv=RUN_STATS_ROOT / "af_mcso_source_promotions_20_to_40.csv",
        stage_summary_csv=RUN_STATS_ROOT / "af_mcso_source_stage_summary.csv",
    )
    selection_seconds = time.perf_counter() - selection_start
    budget = validate_fresh_af_mcso_budget(search_rows)
    save_json(selected_config, RUN_STATS_ROOT / "selected_af_mcso_config.json")
    save_rows_to_csv_safe(build_boundary_report(selected_config),
                          RUN_STATS_ROOT / "selected_config_boundary_report.csv")
    plot_af_mcso_diagnostics(search_rows=search_rows, promotion_rows=promotion_rows,
                             stage_summary_rows=stage_summary_rows)

    final_start = time.perf_counter()
    final_rows, per_class_rows = [], []
    for seed in FINAL_EVALUATION_SEEDS:
        print(f"FINAL AF-MCSO {CROSS_YEAR_TAG} seed={seed}: source train/val, target test")
        start = time.perf_counter()
        cross, source, target, info = make_cross_year_splits_for_seed(seed, source_records, target_records)
        final_yaml, _ = prepare_cross_year_yolo_dataset(
            cross_year_splits=cross, source_splits=source, target_splits=target,
            split_info=info, output_root=YOLO_DATA_ROOT / f"final_seed{seed}",
            manifest_root=SPLITS_ROOT / CROSS_YEAR_TAG / f"final_seed{seed}",
            image_mode=IMAGE_PLACEMENT_MODE, seed=seed,
        )
        train = train_and_validate_configuration(
            **{k: float(selected_config[k]) for k in SEARCH_PARAMETER_ORDER},
            dataset_yaml=final_yaml, project_dir=FINAL_TRAINING_ROOT,
            model_name=MODEL_NAME, epochs=FINAL_EPOCHS, image_size=IMAGE_SIZE,
            batch_size=BATCH_SIZE, patience=PATIENCE, workers=WORKERS, seed=int(seed),
            warmup_epochs=FINAL_WARMUP_EPOCHS, close_mosaic=FINAL_CLOSE_MOSAIC,
            fixed_training_args=FIXED_TRAINING_ARGS, expected_class_names=CLASS_NAMES,
            strict_fitness_alignment_check=STRICT_FITNESS_ALIGNMENT_CHECK,
            fitness_alignment_tolerance=FITNESS_ALIGNMENT_TOLERANCE, device=device,
        )
        target_test = evaluate_checkpoint_on_test_split(
            checkpoint_path=resolve_repo_path(train["paths"]["best_checkpoint"]),
            dataset_yaml=final_yaml, output_root=TEST_EVAL_ROOT / f"seed{seed}",
            run_name=f"{CROSS_YEAR_TAG}_seed{seed}_target_test",
            expected_class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
            workers=WORKERS, device=device,
        )
        row = {
            "status": "PASS", "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR, "test_year": TEST_YEAR, "seed": seed,
            "execution_mode": "fresh_training", "selected_candidate_id": selected_config["selected_candidate_id"],
            **{k: float(selected_config[k]) for k in SEARCH_PARAMETER_ORDER},
            "source_train_count": len(cross["train"]),
            "source_valid_count": len(cross["valid"]),
            "target_test_count": len(cross["test"]),
            "subsample_enabled": bool(info["subsample_info"]["enabled"]),
            "subsample_seed": info["subsample_info"]["subsample_seed"],
            "final_source_val_precision": float(train["metrics"]["precision"]),
            "final_source_val_recall": float(train["metrics"]["recall"]),
            "final_source_val_map50": float(train["metrics"]["map50"]),
            "final_source_val_map50_95": float(train["metrics"]["map50_95"]),
            "final_source_val_bottom3_mean_map50_95": float(train["metrics"]["bottom3_mean_map50_95"]),
            "target_test_precision": float(target_test["test_precision"]),
            "target_test_recall": float(target_test["test_recall"]),
            "target_test_map50": float(target_test["test_map50"]),
            "target_test_map50_95": float(target_test["test_map50_95"]),
            "target_test_bottom3_mean_map50_95": float(target_test["test_bottom3_mean_map50_95"]),
            "target_test_seconds": float(target_test["test_seconds"]),
            "final_training_seconds": float(train["timing"]["training_seconds"]),
            "final_source_validation_seconds": float(train["timing"]["validation_seconds"]),
            "final_run_wall_seconds": float(time.perf_counter() - start),
            "best_checkpoint": train["paths"]["best_checkpoint"],
            "dataset_yaml": portable_path(final_yaml),
        }
        final_rows.append(row)
        per_class_rows.extend({"cross_year_tag": CROSS_YEAR_TAG, "seed": seed, **r}
                              for r in target_test["per_class_rows"])
        save_rows_to_csv_safe(final_rows, RUN_STATS_ROOT / "final_three_run_cross_year_results.csv")
        save_rows_to_csv_safe(per_class_rows, RUN_STATS_ROOT / "final_three_run_target_test_per_class.csv")

    if len(final_rows) != len(FINAL_EVALUATION_SEEDS):
        raise RuntimeError("Final AF-MCSO cross-year test is missing seeded runs")
    summaries = build_af_mcso_target_summary(
        final_rows, new_hpo_units=PLANNED_HPO_EPOCH_UNITS,
        new_final_units=FINAL_EVALUATION_EPOCH_UNITS,
    )
    save_rows_to_csv_safe(summaries, RUN_STATS_ROOT / "final_three_run_target_test_mean_sd.csv")
    save_rows_to_csv_safe(summarize_per_class_rows(per_class_rows),
                          RUN_STATS_ROOT / "final_three_run_target_test_per_class_mean_sd.csv")
    save_target_test_plot(final_rows)
    manifest = {
        "method": "AF-MCSO-600 cross-year fresh training",
        "direction": CROSS_YEAR_TAG, "execution_mode": "fresh_training",
        "source_dataset": portable_path(TRAIN_SOURCE_ROOT),
        "target_dataset": portable_path(TEST_TARGET_ROOT),
        "source_train_valid_only_for_selection": True,
        "target_test_used_during_hpo": False,
        "subsample_year2021_source": bool(TRAIN_YEAR == "2021" and SUBSAMPLE_YEAR2021_TRAIN_IF_SOURCE),
        "audit": audit, "optimization_split_info": optimization_info,
        "search_space": SEARCH_SPACE, "search_space_version": SEARCH_SPACE_VERSION,
        "fixed_training_args": FIXED_TRAINING_ARGS,
        "budget": budget, "final_training_epoch_units": FINAL_EVALUATION_EPOCH_UNITS,
        "final_evaluation_seeds": FINAL_EVALUATION_SEEDS,
        "selected_config": selected_config,
        "selection_stage_wall_seconds": selection_seconds,
        "final_stage_wall_seconds": time.perf_counter() - final_start,
        "workflow_wall_seconds": time.perf_counter() - workflow_start,
        "checkpoint_paths": {str(r["seed"]): r["best_checkpoint"] for r in final_rows},
    }
    save_json(manifest, RUN_STATS_ROOT / "experiment_manifest.json")
    return {
        "final_rows": final_rows, "summary_rows": summaries,
        "selected_config": selected_config, "output_root": portable_path(RUN_ROOT),
        "search_rows": search_rows, "manifest": manifest,
        "execution_mode": "fresh_training", "selection_stage_seconds": selection_seconds,
        "new_hpo_epoch_units": PLANNED_HPO_EPOCH_UNITS,
        "new_final_training_epoch_units": FINAL_EVALUATION_EPOCH_UNITS,
    }


## Module 12: Verified Year2022 single-year checkpoint reuse and automatic direction controller

The notebook probes the original AF-MCSO single-year directory, or the user-specified override. Reuse requires the **full** 600-unit search and **all three final checkpoints**, exact model and protocol, matching source split filenames for each seed, and verified checkpoint class order. Missing/incompatible checkpoints trigger a **fresh** 2022→2021 AF-MCSO search. The first direction is always fresh because its Year2021 training set is downsampled. Combined CSVs and provenance are written after each completed direction.


In [ ]:
# Set False to inspect all cells without launching GPU work.
RUN_CROSS_YEAR_AF_MCSO = False
# Remove only temporary YOLO exports and HPO trial weights *after each successful direction*.
# Never delete final seed weights, source split manifests, or original single-year runs.
CLEANUP_TEMP_FILES_AFTER_SUCCESS = True


def _read_csv_rows(path: Path) -> List[Dict[str, str]]:
    with Path(path).open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def _compatible_af_single_year_run(run_root: Path) -> Dict[str, Any]:
    """Check manifest, full search, all final seed rows, checkpoints and split files."""
    run_root = Path(run_root).expanduser().resolve()
    manifest_file = run_root / "stats" / "experiment_manifest.json"
    hpo_file = run_root / "stats" / "af_mcso_validation_results.csv"
    final_file = run_root / "stats" / "final_three_run_test_results.csv"
    selected_file = run_root / "stats" / "selected_af_mcso_config.json"
    for required in (manifest_file, hpo_file, final_file, selected_file):
        if not required.is_file():
            raise FileNotFoundError(f"Single-year checkpoint evidence missing: {required}")
    manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
    selected = json.loads(selected_file.read_text(encoding="utf-8"))
    if manifest.get("dataset_tag") != "Year2022":
        raise ValueError("Single-year manifest does not describe Year2022")
    if manifest.get("method") != "Adaptive Fidelity-Aware Modified Cuckoo Search (AF-MCSO-600)":
        raise ValueError("Single-year result is not the expected AF-MCSO-600 version")
    runtime, budget, fidelity, mcso = (manifest.get("runtime") or {}, manifest.get("budget") or {},
                                      manifest.get("fidelity") or {}, manifest.get("af_mcso") or {})
    expectations = (
        (runtime.get("model_name"), MODEL_NAME, "model"),
        (int(runtime.get("image_size", -1)), IMAGE_SIZE, "image size"),
        (int(runtime.get("batch_size", -1)), BATCH_SIZE, "batch size"),
        (int(runtime.get("patience", -1)), PATIENCE, "patience"),
        (int(runtime.get("optimization_split_seed", -1)), OPTIMIZATION_SPLIT_SEED, "split seed"),
        (int(runtime.get("optimization_train_seed", -1)), OPTIMIZATION_TRAIN_SEED, "HPO train seed"),
        (int(budget.get("hpo_epoch_units", -1)), PLANNED_HPO_EPOCH_UNITS, "HPO budget"),
        (int(budget.get("expected_low_fidelity_evaluations", -1)), EXPECTED_LOW_FIDELITY_EVALUATIONS, "low evaluations"),
        (int(budget.get("expected_full_fidelity_evaluations", -1)), EXPECTED_FULL_FIDELITY_EVALUATIONS, "full evaluations"),
        (int(budget.get("expected_total_hpo_evaluations", -1)), EXPECTED_TOTAL_HPO_EVALUATIONS, "HPO evaluations"),
        (int(budget.get("final_evaluation_epoch_units", -1)), FINAL_EVALUATION_EPOCH_UNITS, "final units"),
        (int(fidelity.get("low_epochs", -1)), LOW_FIDELITY_EPOCHS, "low fidelity"),
        (int(fidelity.get("full_epochs", -1)), FULL_FIDELITY_EPOCHS, "full fidelity"),
        (int(mcso.get("generations", -1)), MCSO_GENERATIONS, "generations"),
        (int(mcso.get("mcso_random_seed", -1)), MCSO_RANDOM_SEED, "MCSO random seed"),
        (int(mcso.get("final_global_rescue_promotions", -1)), FINAL_RESCUE_PROMOTIONS, "rescue count"),
    )
    for actual, expected, name in expectations:
        if actual != expected:
            raise ValueError(f"Single-year {name} mismatch: {actual} != {expected}")
    if [int(x) for x in runtime.get("final_evaluation_seeds", [])] != FINAL_EVALUATION_SEEDS:
        raise ValueError("Single-year final seeds differ")
    if manifest.get("search_space_version") != SEARCH_SPACE_VERSION or manifest.get("search_space") != SEARCH_SPACE:
        raise ValueError("Single-year AF-MCSO 5D search space differs")
    if manifest.get("fixed_training_args") != FIXED_TRAINING_ARGS:
        raise ValueError("Single-year fixed YOLO training arguments differ")
    if mcso.get("generation_step_scales") != GENERATION_STEP_SCALES:
        raise ValueError("Single-year AF-MCSO step schedule differs")
    if any(float(fidelity.get(k, -1)) != float(v) for k, v in (
        ("low_warmup_epochs", LOW_FIDELITY_WARMUP_EPOCHS),
        ("full_warmup_epochs", FULL_FIDELITY_WARMUP_EPOCHS),
        ("low_close_mosaic", LOW_FIDELITY_CLOSE_MOSAIC),
        ("full_close_mosaic", FULL_FIDELITY_CLOSE_MOSAIC))):
        raise ValueError("Single-year fidelity training controls differ")
    if manifest.get("selected_config") != selected:
        raise ValueError("Single-year saved winner does not match experiment manifest")

    hpo_rows = _read_csv_rows(hpo_file)
    validate_fresh_af_mcso_budget(hpo_rows)  # check actual 600 units, not just planned budget
    full_rows = [r for r in hpo_rows if int(r["budget_epochs"]) == FULL_FIDELITY_EPOCHS]
    if not any(r["candidate_id"] == str(selected.get("selected_candidate_id")) and
               abs(float(r["val_map50_95"]) - float(selected["selected_validation_map50_95"])) < 1e-8
               for r in full_rows):
        raise ValueError("Selected candidate lacks matching observed 40-epoch HPO validation score")

    final_rows = _read_csv_rows(final_file)
    if len(final_rows) != len(FINAL_EVALUATION_SEEDS) or sorted(int(r["seed"]) for r in final_rows) != sorted(FINAL_EVALUATION_SEEDS):
        raise ValueError("Incomplete 3-seed single-year final results CSV")
    saved_splits_root = (Path(YEAR2022_SINGLE_YEAR_SPLITS_OVERRIDE).expanduser().resolve()
                         if YEAR2022_SINGLE_YEAR_SPLITS_OVERRIDE is not None
                         else run_root.parents[2] / "splits" / "Year2022")
    checkpoints, split_manifests = {}, {}
    for row in final_rows:
        seed = int(row["seed"])
        if str(row["selected_candidate_id"]) != str(selected["selected_candidate_id"]):
            raise ValueError(f"Seed {seed} used different selected hyperparameters")
        for parameter in SEARCH_PARAMETER_ORDER:
            if not math.isclose(float(row[parameter]), float(selected[parameter]), rel_tol=1e-9, abs_tol=1e-12):
                raise ValueError(f"Seed {seed} trained with different {parameter}")
        path_text = str(row.get("best_checkpoint", "")).strip()
        if not path_text:
            raise FileNotFoundError(f"Seed {seed} has no recorded checkpoint path")
        checkpoint = resolve_repo_path(path_text)
        # Handle prior experiments moved to another drive when an override points at the run.
        if not checkpoint.is_file():
            original_name = Path(path_text).name
            possible = list((run_root / "final40").glob(f"Y2022_s{seed}_e40_*/weights/{original_name}"))
            if len(possible) == 1:
                checkpoint = possible[0]
        if not checkpoint.is_file() or checkpoint.stat().st_size <= 0:
            raise FileNotFoundError(f"Seed {seed} frozen best.pt not found: {checkpoint}")
        trial_summary = checkpoint.parent.parent / "trial_summary.json"
        if not trial_summary.is_file():
            raise FileNotFoundError(f"Seed {seed} trial training summary missing: {trial_summary}")
        trial = json.loads(trial_summary.read_text(encoding="utf-8"))
        if int(trial.get("hyperparameters", {}).get("epochs", -1)) != FINAL_EPOCHS or int(trial.get("hyperparameters", {}).get("seed", -1)) != seed:
            raise ValueError(f"Seed {seed} final model was not trained for {FINAL_EPOCHS} epochs")
        split_manifest = saved_splits_root / f"final_seed{seed}" / "split_manifest.json"
        if not split_manifest.is_file():
            raise FileNotFoundError(f"Seed {seed} source split manifest missing: {split_manifest}")
        checkpoints[seed] = checkpoint.resolve()
        split_manifests[seed] = split_manifest.resolve()
    return {"run_root": run_root, "manifest": manifest, "selected_config": selected,
            "checkpoints": checkpoints, "split_manifests": split_manifests,
            "historical_selection_seconds": float((manifest.get("timing") or {}).get("selection_stage_wall_seconds", 0.0))}


def locate_completed_year2022_af_mcso() -> Dict[str, Any]:
    """Find newest complete single-year AF-MCSO 600 run or use explicit override."""
    if YEAR2022_SINGLE_YEAR_AF_MCSO_RUN_OVERRIDE:
        return _compatible_af_single_year_run(Path(YEAR2022_SINGLE_YEAR_AF_MCSO_RUN_OVERRIDE))
    base = REPO_ROOT / "outputs" / "YOLOv8m_SingleYear_AF_MCSO_Y2022" / "runs" / "af_mcso20to40_600"
    reasons = []
    for candidate in sorted((x for x in base.iterdir() if x.is_dir()), reverse=True) if base.is_dir() else []:
        try:
            return _compatible_af_single_year_run(candidate)
        except (OSError, ValueError, TypeError, RuntimeError, KeyError, IndexError) as exc:
            reasons.append(f"{candidate.name}: {sanitize_exception_message(exc)}")
    raise FileNotFoundError(f"No complete compatible AF-MCSO single-year Year2022 run in {base}. "
                            + " | ".join(reasons[:5]))


def assert_same_source_splits(source_splits: Dict[str, List[Dict[str, Any]]],
                              seed: int, original: Path) -> None:
    """Compare *every* full-source split, not only train, with the saved original."""
    manifest = json.loads(Path(original).read_text(encoding="utf-8"))
    if int(manifest.get("seed", -1)) != int(seed):
        raise ValueError(f"Seed mismatch in original source manifest seed={seed}")
    for name in ("train", "valid", "test"):
        old = (manifest.get("splits") or {}).get(name, {}).get("filenames")
        fresh = [r["file_name"] for r in source_splits[name]]
        if not isinstance(old, list) or len(old) != len(fresh) or set(old) != set(fresh):
            raise ValueError(f"Source Year2022 split mismatch for seed={seed}: {name}")


def assert_af_checkpoint_class_order(checkpoint: Path) -> None:
    model = YOLO(str(checkpoint))
    names = model.names
    actual = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
    if actual != CLASS_NAMES:
        raise RuntimeError(f"Class names/order in {checkpoint.name} differ: {actual}")
    del model


def probe_af_mcso_year2022_checkpoints() -> Tuple[Optional[Dict[str, Any]], str]:
    """Probe ALL three weights and all source splits before any expensive training."""
    if FORCE_FRESH_2022_TO_2021:
        return None, "FORCE_FRESH_2022_TO_2021=True"
    try:
        saved = locate_completed_year2022_af_mcso()
        records, _ = build_clean_records(DATASET_SOURCE_BY_YEAR["2022"], TARGET_CLASS_SET, LABEL_ALIASES)
        for seed in FINAL_EVALUATION_SEEDS:
            source = split_records_deterministically(records, TRAIN_RATIO, VALID_RATIO, TEST_RATIO, seed)
            assert_same_source_splits(source, seed, saved["split_manifests"][seed])
            assert_af_checkpoint_class_order(saved["checkpoints"][seed])
        return saved, "Complete 600-unit AF-MCSO-600 run, three weights, classes and per-seed source splits verified"
    except (OSError, ValueError, TypeError, RuntimeError, KeyError, IndexError) as exc:
        reason = "Checkpoint reuse unavailable: " + sanitize_exception_message(exc)
        print(reason)
        print("FALLBACK: train 2022->2021 from yolov8m.pt; 600 HPO + 120 final training units")
        return None, reason


def run_reused_af_mcso_2022_to_2021(prior: Dict[str, Any]) -> Dict[str, Any]:
    """Target-year inference ONLY; model.train() is never called."""
    if (TRAIN_YEAR, TEST_YEAR) != ("2022", "2021"):
        raise ValueError("Year2022 single-year checkpoints can only be reused for 2022->2021")
    ensure_output_directories()
    source_records, target_records, audit = scan_cross_year_records()
    started = time.perf_counter()
    device = "0" if torch.cuda.is_available() else "cpu"
    final_rows, per_class_rows = [], []
    selected = prior["selected_config"]
    for seed in FINAL_EVALUATION_SEEDS:
        cross, source, target, info = make_cross_year_splits_for_seed(seed, source_records, target_records)
        assert_same_source_splits(source, seed, prior["split_manifests"][seed])
        checkpoint = prior["checkpoints"][seed]
        assert_af_checkpoint_class_order(checkpoint)
        final_yaml, _ = prepare_cross_year_yolo_dataset(
            cross_year_splits=cross, source_splits=source, target_splits=target,
            split_info=info, output_root=YOLO_DATA_ROOT / f"reuse_seed{seed}",
            manifest_root=SPLITS_ROOT / CROSS_YEAR_TAG / f"reuse_seed{seed}",
            image_mode=IMAGE_PLACEMENT_MODE, seed=seed,
        )
        result = evaluate_checkpoint_on_test_split(
            checkpoint_path=checkpoint, dataset_yaml=final_yaml,
            output_root=TEST_EVAL_ROOT / f"seed{seed}",
            run_name=f"{CROSS_YEAR_TAG}_seed{seed}_frozen_target_test",
            expected_class_names=CLASS_NAMES, image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
            workers=WORKERS, device=device,
        )
        row = {
            "status": "PASS", "cross_year_tag": CROSS_YEAR_TAG,
            "train_year": TRAIN_YEAR, "test_year": TEST_YEAR, "seed": seed,
            "execution_mode": "checkpoint_reuse", "selected_candidate_id": selected["selected_candidate_id"],
            **{k: float(selected[k]) for k in SEARCH_PARAMETER_ORDER},
            "source_train_count": len(cross["train"]), "source_valid_count": len(cross["valid"]),
            "target_test_count": len(cross["test"]),
            "subsample_enabled": False, "subsample_seed": None,
            "target_test_precision": float(result["test_precision"]),
            "target_test_recall": float(result["test_recall"]),
            "target_test_map50": float(result["test_map50"]),
            "target_test_map50_95": float(result["test_map50_95"]),
            "target_test_bottom3_mean_map50_95": float(result["test_bottom3_mean_map50_95"]),
            "target_test_seconds": float(result["test_seconds"]),
            "best_checkpoint": portable_path(checkpoint),
            "dataset_yaml": portable_path(final_yaml),
            "new_hpo_epoch_units": 0, "new_final_training_epoch_units": 0,
        }
        final_rows.append(row)
        per_class_rows.extend({"cross_year_tag": CROSS_YEAR_TAG, "seed": seed, **r}
                              for r in result["per_class_rows"])
        save_rows_to_csv_safe(final_rows, RUN_STATS_ROOT / "reused_af_mcso_target_test_seeds.csv")
        save_rows_to_csv_safe(per_class_rows, RUN_STATS_ROOT / "reused_af_mcso_target_test_per_class.csv")
    summaries = build_af_mcso_target_summary(final_rows, new_hpo_units=0, new_final_units=0,
                                            historical_selection_seconds=prior["historical_selection_seconds"])
    save_rows_to_csv_safe(summaries, RUN_STATS_ROOT / "reused_af_mcso_target_test_mean_sd.csv")
    save_rows_to_csv_safe(summarize_per_class_rows(per_class_rows),
                          RUN_STATS_ROOT / "reused_af_mcso_target_test_per_class_mean_sd.csv")
    save_target_test_plot(final_rows)
    save_json({
        "method": "AF-MCSO-600 cross-year test-only checkpoint reuse",
        "direction": CROSS_YEAR_TAG, "execution_mode": "checkpoint_reuse",
        "original_run": portable_path(prior["run_root"]),
        "original_hpo_epoch_units": PLANNED_HPO_EPOCH_UNITS,
        "historical_selection_stage_wall_seconds": prior["historical_selection_seconds"],
        "new_hpo_epoch_units": 0, "new_final_training_epoch_units": 0,
        "source_split_manifests_verified": True, "checkpoint_class_order_verified": True,
        "target_test_only": True,
        "seed_checkpoints": {str(k): portable_path(v) for k, v in prior["checkpoints"].items()},
        "target_test_wall_seconds": time.perf_counter() - started,
        "audit": audit,
    }, RUN_STATS_ROOT / "checkpoint_reuse_provenance.json")
    return {
        "final_rows": final_rows, "summary_rows": summaries,
        "selected_config": selected, "output_root": portable_path(RUN_ROOT),
        "execution_mode": "checkpoint_reuse",
        "selection_stage_seconds": prior["historical_selection_seconds"],
        "historical_checkpoint_run": portable_path(prior["run_root"]),
        "new_hpo_epoch_units": 0, "new_final_training_epoch_units": 0,
    }


def paper_result_row_af_mcso(direction: str, result: Dict[str, Any]) -> Dict[str, Any]:
    final_rows = result["final_rows"]
    if len(final_rows) != len(FINAL_EVALUATION_SEEDS):
        raise RuntimeError(f"Incomplete target tests for {direction}")
    def fmt(key: str) -> str:
        vals = [100.0 * float(row[key]) for row in final_rows]
        return f"{mean(vals):.2f} ± {stdev(vals):.2f}"
    return {
        "Direction": direction, "Method": "AF-MCSO-600",
        "Precision": fmt("target_test_precision"),
        "Recall": fmt("target_test_recall"),
        "mAP50": fmt("target_test_map50"),
        "mAP50_95": fmt("target_test_map50_95"),
        "Optimization Time (s)": round(float(result["selection_stage_seconds"]), 2),
        "HPO time origin": ("historical single-year Year2022" if result["execution_mode"] == "checkpoint_reuse"
                            else "new cross-year source-validation search"),
        "Execution mode": result["execution_mode"],
        "New HPO epoch-units": int(result["new_hpo_epoch_units"]),
        "New final training epoch-units": int(result["new_final_training_epoch_units"]),
        "Historical HPO epoch-units": (PLANNED_HPO_EPOCH_UNITS if result["execution_mode"] == "checkpoint_reuse" else 0),
    }


def run_automatic_cross_year_af_mcso() -> Dict[str, Any]:
    """Preflight two datasets and checkpoint set, then run both transfer directions."""
    global DATASET_SOURCE_BY_YEAR
    DATASET_SOURCE_BY_YEAR = resolve_source_years()
    if (PLANNED_HPO_EPOCH_UNITS, EXPECTED_LOW_FIDELITY_EVALUATIONS,
        EXPECTED_FULL_FIDELITY_EVALUATIONS, EXPECTED_TOTAL_HPO_EVALUATIONS) != (600, 14, 8, 22):
        raise RuntimeError("AF-MCSO-600 schedule changed unexpectedly")
    prior, decision = probe_af_mcso_year2022_checkpoints()
    print("Year2022 checkpoint decision:", "REUSE" if prior else "TRAIN FRESH", "|", decision)

    COMBINED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    paper_csv = COMBINED_OUTPUT_ROOT / "af_mcso_cross_year_paper_ready_test_results.csv"
    seeds_csv = COMBINED_OUTPUT_ROOT / "af_mcso_cross_year_all_seed_test_results.csv"
    summary_csv = COMBINED_OUTPUT_ROOT / "af_mcso_cross_year_mean_sd_summary.csv"
    selected_csv = COMBINED_OUTPUT_ROOT / "af_mcso_cross_year_selected_hyperparameters.csv"
    combined_json = COMBINED_OUTPUT_ROOT / "af_mcso_cross_year_combined_manifest.json"
    completed, paper_rows, all_seeds, all_summary, all_selected = {}, [], [], [], []

    def persist(direction: str, result: Dict[str, Any]) -> None:
        completed[direction] = result
        paper_rows.append(paper_result_row_af_mcso(direction, result))
        all_seeds.extend({"direction": direction, **r} for r in result["final_rows"])
        all_summary.extend({"direction": direction, **r} for r in result["summary_rows"])
        all_selected.append({"direction": direction, "execution_mode": result["execution_mode"],
                             **result["selected_config"]})
        save_rows_to_csv_safe(paper_rows, paper_csv)
        save_rows_to_csv_safe(all_seeds, seeds_csv)
        save_rows_to_csv_safe(all_summary, summary_csv)
        save_rows_to_csv_safe(all_selected, selected_csv)
        save_json({
            "method": "AF-MCSO-600 cross-year automatic reuse-or-train",
            "directions_requested": ["2021_to_2022", "2022_to_2021"],
            "directions_completed": list(completed),
            "checkpoint_probe": decision,
            "execution_modes": {d: r["execution_mode"] for d, r in completed.items()},
            "new_hpo_epoch_units": sum(r["new_hpo_epoch_units"] for r in completed.values()),
            "new_final_training_epoch_units": sum(r["new_final_training_epoch_units"] for r in completed.values()),
            "single_direction_fresh_hpo_units": PLANNED_HPO_EPOCH_UNITS,
            "single_direction_fresh_final_units": FINAL_EVALUATION_EPOCH_UNITS,
            "historical_hpo_time_is_not_new_runtime": True,
            "year2021_source_train_balancing": "stratified to Year2022 train size",
            "target_test_used_for_hpo": False,
            "final_seeds": FINAL_EVALUATION_SEEDS,
            "per_direction_runs": {d: r["output_root"] for d, r in completed.items()},
            "paper_ready_results": portable_path(paper_csv),
        }, combined_json)
        if CLEANUP_TEMP_FILES_AFTER_SUCCESS:
            for temporary in (YOLO_DATA_ROOT, HPO_RUNS_ROOT):
                if temporary.is_dir():
                    shutil.rmtree(temporary)
            print("Removed only the current cross-year direction temporary YOLO/HPO exports")
        print("Completed:", direction, "| mode:", result["execution_mode"])

    activate_cross_year_direction("2021", "2022")
    forward = run_fresh_cross_year_af_mcso()  # balancing -> cannot reuse single-year Year2021
    persist("2021_to_2022", forward)

    activate_cross_year_direction("2022", "2021")
    backward = (run_reused_af_mcso_2022_to_2021(prior) if prior is not None
                else run_fresh_cross_year_af_mcso())
    persist("2022_to_2021", backward)
    print("Both cross-year AF-MCSO transfers complete.")
    print("New HPO units:", sum(r["new_hpo_epoch_units"] for r in completed.values()))
    print("New final training units:", sum(r["new_final_training_epoch_units"] for r in completed.values()))
    print("Paper-ready results:", paper_csv)
    return {"results": completed, "paper_ready_csv": str(paper_csv),
            "seeds_csv": str(seeds_csv), "summary_csv": str(summary_csv),
            "selected_csv": str(selected_csv), "manifest": str(combined_json)}


if RUN_CROSS_YEAR_AF_MCSO:
    cross_year_af_mcso_result = run_automatic_cross_year_af_mcso()
else:
    print("RUN_CROSS_YEAR_AF_MCSO=False: no GPU training or target testing started")
    print("Directions:", DIRECTIONS)
    print("If Year2022 reuse succeeds: 600 new HPO + 120 new final epoch-units")
    print("If reuse is unavailable: 1200 new HPO + 240 new final epoch-units")


## Reproducibility, non-leakage and checkpoint-reuse notes

- **Unchanged AF-MCSO optimizer:** Five generations, two 20-epoch proposals per generation, one 40-epoch promotion per generation, final 40-epoch archive rescue, and a fully measured 40-epoch winning configuration. Total exactly **600 HPO epoch-units per fresh direction**.
- **Balanced Year2021 source (DEHB-matched):** Source Year2021 training is stratified-subsampled to Year2022's 70% training split size using `seed + 10000`. Validation remains full Year2021 validation; testing is Year2022 test only. For reverse transfer, Year2022 source training uses its complete train split.
- **Target isolation:** AF-MCSO and final checkpoint selection see source training/validation only. Target test is used once per final seed for cross-year reporting; the held-out target training/validation portions never enter HPO.
- **Reuse only when verified:** Both 600-unit HPO manifests and successful logs, all three 40-epoch single-year final runs, their matching selected hyperparameters, actual `best.pt` weights, trial summaries, original source filename split manifests, and checkpoint class names must match. If any check fails, the reverse direction trains from scratch. Only 2022 → 2021 can reuse *single-year* weights because 2021 → 2022 uses the balanced training subset.
- **No accidental deletion:** Cleanup removes generated *cross-year* temporary YOLO/HPO data for the just-completed direction, not final seed checkpoints, split manifests, source datasets, or source single-year runs.
- **Budget reporting:** Checkpoint reuse counts 0 **new** HPO / 0 **new** final epoch-units; the historical 600 HPO units are explicitly distinguished from current-run spending. Fresh training counts 600 + 120 epoch-units per direction. Epoch-units are the sum of requested training-epoch budgets across evaluations, not wall-clock epochs or a continued single-model run.
- **Environment:** Requires working Ultralytics/PyTorch/PIL/NumPy/Matplotlib environment and original dataset layouts. The supplied artifact is not evidence of GPU training, final scores, or checkpoint availability on the user's PC.
- **For moved historical weights:** Set `YEAR2022_SINGLE_YEAR_AF_MCSO_RUN_OVERRIDE` and, when necessary, `YEAR2022_SINGLE_YEAR_SPLITS_OVERRIDE`. These are run directory / splits directory paths, not individual `.pt` paths.

- **GitHub execution:** Keep `RUN_CROSS_YEAR_AF_MCSO = False` when publishing. Run from your repository root or edit `MAIN_DIRECTORY` to your local project root; the notebook does not contain a fixed machine-specific location. The default combined reports are saved under `outputs/YOLOv8m_CrossYear_AF_MCSO_Auto600/`.
